# Norm Information and Coherence in Quantum MRI Classification
## Patient-level publication study · version 3.1 · runtime repair

### Goal
Test **whether retaining feature norm and enabling ancilla interference improves prediction,
calibration or robustness**, compared with ordinary amplitude encoding and classical models
given the same information. A null result is allowed. No quantum speedup is assumed.

This revision implements the study, not a promise of acceptance. **There are no new scientific
results stored in this download.** Run the real-data profile before writing results. All old
Figshare/Kaggle findings were already inspected: every new internal analysis is explicitly
**exploratory**, including newly randomized splits. Only a separately acquired, previously
unseen cohort can support a new external confirmation claim.

**Runtime repair:** setup verifies compatible imports before training; local Torch installation,
MATLAB reference readers, download/weight validation, cluster-safe splits, batch-loaded CNN
training, UTF-8 reporting and local ZIP downloads have been repaired. Defaults use the real-data
`pilot` profile. The final engineering validation status is recorded in notebook metadata.
There are no precomputed scientific performance claims in this file.

**Validation:** all 40 code cells passed on the real Figshare `pilot` run (3,064 images,
233 patients, three outer folds and three representations). A separate four-class synthetic
run passed all seven representations and one epoch of CNN training on two folds. Corrupt MAT,
MATLAB object references, linked-patient splits, checksum retries, resumed extraction and
UTF-8/Markdown reporting checks passed. Validation used CPU and Python 3.12; the full `paper`
budget, Colab GPU execution and a fresh external cohort were not tested.

### Run order
1. Open in Google Colab. A GPU is recommended for `pilot` and `paper`; CPU is also supported.
2. In Cell 1 choose `smoke` for a fast synthetic engineering check, then `pilot` for real-data setup.
3. Use `paper` for the complete repeated study. Run cells in order. The full study can take
   several GPU/CPU sessions; checkpoints resume completed fits and folds.
4. Mount Drive when prompted. If Drive is unavailable, save the evidence ZIP before disconnecting.
5. Supply the external cohort and its metadata/provenance only when available. The notebook
   reports missing external evidence; it never substitutes Kaggle or synthetic images.

| Change | Implementation |
|---|---|
| One test split | Repeated stratified **patient/unit outer folds**, with separate fit, selection, temperature and conformal patients |
| Trivial calibration | At least 12 calibration patients per class in real runs; LAC and APS sets, set size and singleton reporting |
| Severe compression | 32/64-pixel controls, 16/64 PCA components, shared pretrained ResNet features and an uncompressed probe |
| Weak optimization evidence | Longer training, learning-rate search, gradient checks and convergence logs |
| Acquisition shortcuts | Border-only model, foreground-only sensitivity, raw-intensity sensitivity, fixed perturbations |
| Classical fairness | Norm, direction, norm+direction, quadratic, SVM, MLP, RF, frozen CNN and fine-tuned CNN controls |
| Unverified generalization | External patient/provenance/overlap gate; no external tuning or recalibration |
| Reporting | Fold/OOF predictions, class metrics, effect intervals, resources, manuscript working draft and unresolved-requirement checklist |

### Scope and key assumptions
Each analysis unit has one diagnosis; calibrated probabilities are averaged over its available
slices. A unit is one patient when duplicate links permit, otherwise an explicitly labelled
linked patient cluster that stays intact across all split roles. Native Figshare has three tumor types, not a no-tumor class. Kaggle is an optional
four-class composite benchmark and remains exploratory unless trustworthy IDs are supplied.
Foreground masking is an image-derived heuristic, **not validated skull stripping**.
Class-conditional conformal validity assumes exchangeability within each class; external
scanner/site shifts do not inherit that guarantee. Repeated folds are not independent patients.

### Cell 1 — Configure the study

The default `pilot` profile runs real MRI data with a smaller budget. Set `PROFILE='smoke'`
for a fast synthetic code check; use `paper` after the pilot succeeds. A GPU speeds up ResNet;
CPU is supported. Run **all cells from the top** after changing the profile or any path.
Use native Figshare `.mat` files for verified patient IDs, or supply a class-folder root
plus its patient metadata CSV. Paths can be set below or through `QML_*` environment variables.
To run offline, also set `QML_RESNET_WEIGHTS_PATH` to the official `resnet18-f37072fd.pth`.


In [ ]:
import os, sys, copy
from pathlib import Path
PROFILE = 'pilot'  # @param ['smoke','pilot','paper']
PROFILE = os.getenv('QML_PROFILE', PROFILE).strip().lower()
USE_DRIVE = os.getenv('QML_USE_DRIVE','1') == '1'  # @param {type:'boolean'}
FIGSHARE_ROOT = os.getenv('QML_FIGSHARE_ROOT','')
KAGGLE_ROOT = os.getenv('QML_KAGGLE_ROOT','')
FOLDER_ROOT = os.getenv('QML_FOLDER_ROOT','')
METADATA_CSV = os.getenv('QML_METADATA_CSV','')
INCLUDE_KAGGLE = os.getenv('QML_INCLUDE_KAGGLE','0') == '1'
EXTERNAL_ROOT = os.getenv('QML_EXTERNAL_ROOT','')
EXTERNAL_METADATA_CSV = os.getenv('QML_EXTERNAL_METADATA_CSV','')
EXTERNAL_PROVENANCE_JSON = os.getenv('QML_EXTERNAL_PROVENANCE_JSON','')
RESULTS_ROOT = os.getenv('QML_RESULTS_DIR','/content/qml_mri_v3' if Path('/content').exists() else str(Path.cwd()/'qml_mri_v3'))
DATA_CACHE = os.getenv('QML_DATA_CACHE','/content/qml_data_cache' if Path('/content').exists() else str(Path.cwd()/'qml_data_cache'))
PROFILES = {
 'smoke': dict(folds=2,repeats=1,seeds=[13],epochs=2,warmup=1,patience=2,
     lrs=[0.02],Cs=[1.0],batch=32,min_cf=9,min_temp=2,min_select=2,min_fit=3,
     slice_cap=2,bootstrap=50,permutations=50,cnn_epochs=1,noise_units=3,shot_reps=2),
 'pilot': dict(folds=3,repeats=1,seeds=[13],epochs=15,warmup=5,patience=5,
     lrs=[0.01,0.04],Cs=[0.3,3.0],batch=64,min_cf=12,min_temp=3,min_select=3,min_fit=8,
     slice_cap=4,bootstrap=300,permutations=300,cnn_epochs=3,noise_units=6,shot_reps=3),
 'paper': dict(folds=5,repeats=2,seeds=[13,29,47],epochs=120,warmup=25,patience=20,
     lrs=[0.005,0.02,0.05],Cs=[0.03,0.3,3.0,30.0],batch=64,min_cf=12,min_temp=3,min_select=3,min_fit=8,
     slice_cap=8,bootstrap=3000,permutations=5000,cnn_epochs=30,noise_units=24,shot_reps=20),
}
if PROFILE not in PROFILES: raise ValueError('Choose smoke, pilot or paper.')
CFG=copy.deepcopy(PROFILES[PROFILE])
VERSION='3.1.0'; ALPHA=0.10; N_LAYERS=2; SPLIT_SEED=20261009
PHASH_RADIUS=3; NEAR_SSIM_THRESHOLD=0.985
RUN_FINE_TUNE = os.getenv('QML_FINE_TUNE', '1' if PROFILE=='paper' else '0')=='1' or os.getenv('QML_TEST_CNN','0')=='1'
RUN_NOISE = os.getenv('QML_RUN_NOISE','1')=='1'
RUN_DIAGNOSTICS = os.getenv('QML_RUN_DIAGNOSTICS','1')=='1'
DOWNLOAD_ZIP=False
RESNET_WEIGHTS_PATH=os.getenv('QML_RESNET_WEIGHTS_PATH','').strip()
FEATURE_BATCH_SIZE=16
CNN_BATCH_SIZE=16
CNN_INFERENCE_BATCH_SIZE=16
PRIMARY_SPEC='pixels32_d16' if PROFILE=='smoke' else 'resnet18_d16'
SPEC_LIBRARY={
 'pixels32_d16':dict(view='pixels32',dim=16),
 'pixels64_d16':dict(view='pixels64',dim=16),
 'pixels64_d64':dict(view='pixels64',dim=64),
 'foreground64_d16':dict(view='foreground64',dim=16),
 'raw64_d16':dict(view='raw64',dim=16),
 'resnet18_d16':dict(view='resnet18',dim=16),
 'resnet18_d64':dict(view='resnet18',dim=64),
}
SPECS=list(SPEC_LIBRARY) if PROFILE=='paper' else (
 ['pixels32_d16','pixels64_d16','resnet18_d16'] if PROFILE=='pilot' else ['pixels32_d16'])
COHORTS=['synthetic3'] if PROFILE=='smoke' else (['folder'] if FOLDER_ROOT else ['figshare3'])
if INCLUDE_KAGGLE and PROFILE!='smoke': COHORTS.append('kaggle4')
# Extra representations use the first initialization seed on every outer fold;
# the primary representation uses every seed. This distinction is reported.
CFG['fit_slices_per_unit']=CFG['slice_cap']
MANUAL_REVIEW={
 'novelty_comparison_completed':False,
 'dataset_license_and_ethics_checked':False,
 'duplicate_and_shortcut_review_completed':False,
 'manuscript_and_target_venue_checked':False,
}
if FOLDER_ROOT and not METADATA_CSV:
    raise ValueError('FOLDER_ROOT needs METADATA_CSV with relative_path, patient_id and source_dataset. Use native Figshare MAT files if verified patient metadata is unavailable.')
if EXTERNAL_ROOT and (not EXTERNAL_METADATA_CSV or not EXTERNAL_PROVENANCE_JSON):
    raise ValueError('Set both EXTERNAL_METADATA_CSV and EXTERNAL_PROVENANCE_JSON before enabling EXTERNAL_ROOT.')
if not all(isinstance(x,int) and x>0 for x in [FEATURE_BATCH_SIZE,CNN_BATCH_SIZE,CNN_INFERENCE_BATCH_SIZE]):
    raise ValueError('Batch sizes must be positive integers.')
print('Profile:',PROFILE,'| cohorts:',COHORTS,'| primary:',PRIMARY_SPEC)
print('Internal analyses are exploratory. A profile name does not certify publication.')
print('Use smoke for a fast synthetic code check; pilot and paper use real MRI data.')

### Cell 2 — Install and verify dependencies

Use a fresh Python 3.12 Colab/Jupyter environment. This cell installs missing requirements,
checks native imports, and keeps the pinned quantum stack compatible with an installed JAX.
A working Colab GPU Torch/Torchvision pair is preserved; a missing local pair is installed
for CPU. If setup replaces an **already imported** package, it requests one kernel/session
restart before training. After that restart, choose **Run all**. `QML_SKIP_INSTALL=1` validates
an already prepared environment and reports missing/incompatible dependencies explicitly.


In [ ]:
import subprocess, importlib.metadata, importlib.util
from packaging.requirements import Requirement
from packaging.version import Version

# The NumPy simulator is used for training. PennyLane validates circuits/noise.
# Keep PennyLane, Lightning and Autoray compatible. JAX is not used for training,
# but PennyLane imports it if it is installed (as it usually is in Colab).
requirements=['pennylane==0.42.3','pennylane-lightning==0.42.0','autoray==0.7.2',
 'numpy>=1.26,<2.4','scipy>=1.11,<2','pandas>=2,<3','scikit-learn>=1.6.1,<2',
 'h5py>=3.14,<4','ImageHash==4.3.2','scikit-image>=0.25.2,<0.27',
 'kagglehub==0.3.13','requests>=2.31,<3','matplotlib>=3.7,<4',
 'joblib>=1.3,<2','tabulate>=0.9,<1','ipython>=8']
try:
    importlib.metadata.version('jax')
except importlib.metadata.PackageNotFoundError:
    pass
else:
    requirements += ['jax==0.6.2','jaxlib==0.6.2']

def dependency_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

module_names={'scikit-learn':'sklearn','scikit-image':'skimage','ImageHash':'imagehash',
              'pennylane-lightning':'pennylane_lightning'}
missing=[]; loaded_changes=[]
for requirement in requirements:
    req=Requirement(requirement); installed=dependency_version(req.name)
    if installed is None or installed not in req.specifier:
        missing.append(requirement)
        module=module_names.get(req.name,req.name.replace('-','_'))
        if any(name==module or name.startswith(module+'.') for name in sys.modules):
            loaded_changes.append(req.name)
if missing:
    if os.getenv('QML_SKIP_INSTALL','0')=='1':
        raise RuntimeError('QML_SKIP_INSTALL=1 but the environment is incomplete. Install: '+', '.join(missing))
    print('Installing compatible study dependencies:',', '.join(missing),flush=True)
    try:
        subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Dependency installation failed. Check the pip output above and internet access. Use Python 3.12 in a fresh Colab/Jupyter environment, then rerun from the first cell.') from exc
    if loaded_changes:
        raise RuntimeError('Installation succeeded, but these packages were already imported: '+', '.join(sorted(set(loaded_changes)))+'. In Colab choose Runtime > Restart session, then Run all. In Jupyter restart the kernel and run all cells. No training has started.')

NEEDS_TORCH=PROFILE!='smoke' or RUN_FINE_TUNE
if NEEDS_TORCH:
    pair_missing=dependency_version('torch') is None or dependency_version('torchvision') is None
    if pair_missing:
        if os.getenv('QML_SKIP_INSTALL','0')=='1':
            raise RuntimeError('Install matching torch and torchvision, or use PROFILE="smoke". CPU command: python -m pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu')
        # Preserve a working Colab GPU installation. For a local environment
        # with no complete pair, install a tested CPU pair instead of CUDA GBs.
        loaded_torch=any(n=='torch' or n.startswith(('torch.','torchvision.')) for n in sys.modules)
        print('Installing the local CPU Torch/Torchvision pair...',flush=True)
        try:
            subprocess.check_call([sys.executable,'-m','pip','install','torch==2.8.0','torchvision==0.23.0',
                                   '--index-url','https://download.pytorch.org/whl/cpu'])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError('Torch installation failed. Install the matching CPU pair printed above, restart the kernel, and run all.') from exc
        if loaded_torch:
            raise RuntimeError('Torch/Torchvision were repaired. Restart the session/kernel and Run all before importing them.')
    try:
        import torch, torchvision
        from torchvision.models import resnet18, ResNet18_Weights
        # Imports alone may succeed even when custom Torchvision ops do not.
        torchvision.ops.nms(torch.tensor([[0.,0.,1.,1.]]),torch.tensor([1.]),0.5)
    except Exception as exc:
        raise RuntimeError('Torch/Torchvision are incompatible or their native libraries failed to load. Use a fresh Colab runtime. For local CPU, install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu, restart the kernel, then Run all. Original error: '+str(exc)) from exc
    print('Torch:',torch.__version__,'Torchvision:',torchvision.__version__,'GPU:',torch.cuda.is_available())
    if not torch.cuda.is_available(): print('Using CPU. Pilot is supported; the paper profile will be slow.')
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')

# Check real imports before downloading the dataset or starting a long run.
try:
    import numpy, scipy, pandas, sklearn, h5py, imagehash, skimage
    import pennylane as qml
except Exception as exc:
    raise RuntimeError('A dependency failed to import after setup. Restart the kernel/session and Run all. If it persists, use a fresh Python 3.12 environment. Original error: '+str(exc)) from exc
print('Dependency imports passed. PennyLane:',qml.__version__,'NumPy:',numpy.__version__)


### Cell 3 — Imports, reproducibility and output location

Records versions and the exact implementation hash. GPU determinism is requested;
hardware/library differences may still affect floating-point results. Checkpoints are local
trusted files produced by this notebook; never load untrusted pickle/joblib files.

In [ ]:
import contextlib, hashlib, json, math, random, shutil, time, platform, pickle, zlib, zipfile
import urllib.request, urllib.error, urllib.parse
from collections import Counter, defaultdict
import h5py, imagehash, joblib
import numpy as np
import pandas as pd
import scipy, scipy.io
from scipy.optimize import minimize, minimize_scalar
from scipy.special import softmax, logsumexp
from scipy.ndimage import binary_fill_holes, label as component_label, gaussian_filter
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
from skimage.metrics import structural_similarity
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, confusion_matrix, balanced_accuracy_score,
    log_loss, roc_auc_score, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
import pennylane as qml
import warnings
pd.set_option('display.max_rows',30)
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    if 'torch' in sys.modules:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark=False
        torch.backends.cudnn.deterministic=True
        torch.use_deterministic_algorithms(True,warn_only=True)
        torch.set_num_threads(min(4,os.cpu_count() or 1))
def digest(value):
    def canonical(x):
        if isinstance(x,np.ndarray): return x.tolist()
        if isinstance(x,np.generic): return x.item()
        if isinstance(x,dict): return {str(k):canonical(v) for k,v in x.items()}
        if isinstance(x,(list,tuple)): return [canonical(v) for v in x]
        if isinstance(x,Path): return str(x)
        return x
    return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,default=str).encode()).hexdigest()
def file_sha(path,algorithm='sha256'):
    h=hashlib.new(algorithm)
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def write_json(path,value):
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    Path(path).write_text(json.dumps(value,indent=2,default=str),encoding='utf-8')
def atomic_joblib(value,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp'); joblib.dump(value,temp); temp.replace(path)
def cached(path,identity,build):
    path=Path(path)
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj['identity']!=identity: raise ValueError('Cache identity mismatch')
            return obj['payload']
        except (EOFError,OSError,ValueError,KeyError,TypeError,ImportError,IndexError,AttributeError,pickle.UnpicklingError,zlib.error) as exc:
            path.rename(path.with_suffix(path.suffix+'.invalid.'+str(time.time_ns())))
            print('Quarantined invalid cache:',path.name,type(exc).__name__)
    result=build(); atomic_joblib(dict(identity=identity,payload=result),path); return result

def markdown_table(table):
    try: return table.to_markdown(index=False)
    except ImportError:
        # Reporting still works if a user prepared dependencies manually.
        def clean(value): return str(value).replace('|',r'\|').replace('\n',' ')
        header='| '+' | '.join(map(clean,table.columns))+' |'
        rule='| '+' | '.join(['---']*len(table.columns))+' |'
        rows=['| '+' | '.join(clean(v) for v in row)+' |' for row in table.itertuples(index=False,name=None)]
        return '\n'.join([header,rule,*rows])

seed_everything(SPLIT_SEED)
if USE_DRIVE and PROFILE!='smoke':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        RESULTS_ROOT='/content/drive/MyDrive/QML_MRI_v3' if not os.getenv('QML_RESULTS_DIR') else RESULTS_ROOT
    except ImportError: print('Running outside Colab; results are saved to',RESULTS_ROOT)
    except Exception as exc: print('Drive unavailable:',str(exc),'Save the ZIP before the runtime ends.')
CACHE=Path(DATA_CACHE); CACHE.mkdir(parents=True,exist_ok=True)
ENV={name:importlib.metadata.version(name) for name in ['numpy','scipy','pandas','scikit-learn','pennylane','h5py','ImageHash']}
ENV.update(python=sys.version,platform=platform.platform())
for name in ['autoray','pennylane-lightning','jax','jaxlib','scikit-image','joblib']:
    with contextlib.suppress(importlib.metadata.PackageNotFoundError): ENV[name]=importlib.metadata.version(name)
for name in ['torch','torchvision']:
    with contextlib.suppress(importlib.metadata.PackageNotFoundError): ENV[name]=importlib.metadata.version(name)
CODE_SHA='0df9ade3abfbf1a3a59f3b45464015fc0c527c096a75b1cdd8787990dc961458'
PROTOCOL=dict(version=VERSION,profile=PROFILE,cfg=CFG,primary=PRIMARY_SPEC,specs=SPECS,
    cohorts=COHORTS,seed=SPLIT_SEED,alpha=ALPHA,layers=N_LAYERS,fine_tune=RUN_FINE_TUNE,noise=RUN_NOISE,diagnostics=RUN_DIAGNOSTICS,
    feature_batch=FEATURE_BATCH_SIZE,cnn_batch=CNN_BATCH_SIZE,resnet_weight_source=RESNET_WEIGHTS_PATH or 'IMAGENET1K_V1',
    phash_radius=PHASH_RADIUS,ssim=NEAR_SSIM_THRESHOLD,code_sha=CODE_SHA,environment=ENV,
    endpoint='patient/unit macro-F1',internal_status='exploratory_after_prior_test_inspection',
    contrasts=['Coherent-Passive','Coherent-Amplitude','Coherent-SelectedClassical'])
PROTOCOL_HASH=digest(PROTOCOL)
RUN_ROOT=Path(RESULTS_ROOT)/f'{PROFILE}_{PROTOCOL_HASH[:12]}'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
write_json(RUN_ROOT/'protocol.json',PROTOCOL)
write_json(RUN_ROOT/'environment.json',ENV)
(RUN_ROOT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
print('Results:',RUN_ROOT,'\nProtocol:',PROTOCOL_HASH)

## Context & Methods

**Primary mechanistic comparison:** two-layer circuits with the same trainable rotations on
the data register. Amplitude encoding discards norm; Passive encodes norm in an ancilla whose
interactions preserve a mixture structure; Coherent interleaves fixed Hadamard gates to allow
interference. ConstantNorm and ShuffledNorm isolate dependence on sample norm. At dimension
16 there are 24 trainable angles; at dimension 64 there are 36. All models receive their own
one-parameter temperature fitted on separate patients.

**Primary representation:** frozen ImageNet ResNet18 features, train-only PCA to dimension 16.
This is a deliberate post-review study change, not a retrospectively prespecified discovery.
The original pixel representation remains a sensitivity control. Every quantum/classical
comparison within a representation uses the same patient partitions and fitted PCA.

**Evaluation:** 5 outer folds × 2 repeats for `paper`. Each outer test patient is unseen by
that fold's PCA, training, selection and calibration. Hyperparameters use only the inner
selection patients. Temperature and conformal patients are separate. Extra representations
use one initialization per outer fold; the primary uses three. Internal intervals describe
resampling of the observed OOF predictions; overlapping training sets prevent interpreting
them as independent repeated-experiment confidence intervals. No internal significance
claim or acceptance score is generated.

**Fresh external evaluation:** final models use an independently partitioned development
cohort. Model/representation choices are fixed before external evaluation. Paired patient
permutation tests and Holm correction apply to the three fixed external comparisons only.
Clinical use, scanner generalization and quantum computational advantage require evidence
beyond this small simulated retrospective study.

### Cell 4 — Download and archive validation

Figshare version 8 is fixed and its publisher checksum is verified. Downloads never fall back
to synthetic data. Supply local paths in Cell 1 if public downloads are unavailable.

In [ ]:
def get_json(url, attempts=3):
    for attempt in range(attempts):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "QML-MRI-Study/3.1.0"})
            with urllib.request.urlopen(request, timeout=60) as response:
                return json.load(response)
        except (OSError, ValueError) as exc:
            if attempt + 1 == attempts:
                raise RuntimeError(f"Could not read {url}. Check internet access or supply a local dataset root.") from exc
            print(f"Metadata retry {attempt + 1}/{attempts}: {type(exc).__name__}", flush=True)
            time.sleep(2 ** attempt)

def download_url(url, destination, expected_md5=None, attempts=4, expected_sha256_prefix=None):
    import requests
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and destination.stat().st_size > 0:
        if (not expected_md5 or file_sha(destination, "md5") == expected_md5) and (not expected_sha256_prefix or file_sha(destination).startswith(expected_sha256_prefix)):
            return destination
    temp = destination.with_suffix(destination.suffix + ".part")
    session = requests.Session()
    for attempt in range(attempts):
        try:
            # Figshare redirects to S3 URLs with short expiry. A fresh query and
            # a reused connection avoid cached/expired redirects on retries.
            request_url = url
            if (urllib.parse.urlsplit(url).hostname or "").endswith("figshare.com"):
                request_url += ("&" if "?" in url else "?") + "qml_cachebust=" + str(time.time_ns())
            print("Downloading", destination.name, flush=True)
            with session.get(request_url, stream=True, timeout=(20, 90),
                    headers={"User-Agent": "QML-MRI-Study/3.1.0", "Cache-Control": "no-cache"}) as response, open(temp, "wb") as handle:
                response.raise_for_status()
                expected_length = response.headers.get("Content-Length")
                if response.headers.get("Content-Encoding"):
                    expected_length = None
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
            if temp.stat().st_size == 0 or (expected_length and temp.stat().st_size != int(expected_length)):
                raise OSError("Download is empty or truncated.")
            if expected_md5 and file_sha(temp, "md5") != expected_md5:
                raise ValueError("Publisher MD5 mismatch: " + destination.name)
            if expected_sha256_prefix and not file_sha(temp).startswith(expected_sha256_prefix):
                raise ValueError("Publisher SHA256 mismatch: "+destination.name)
            temp.replace(destination)
            return destination
        except (requests.RequestException, OSError, ValueError) as exc:
            temp.unlink(missing_ok=True)
            if attempt + 1 == attempts:
                raise RuntimeError(f"Download failed for {destination.name}. Correct internet access or set FIGSHARE_ROOT to an extracted native dataset in Cell 1, then rerun from the first cell.") from exc
            status = getattr(getattr(exc, "response", None), "status_code", None)
            print(f"Download retry {attempt + 1}/{attempts}: {type(exc).__name__}"
                  + (f" (HTTP {status}; requesting a fresh publisher link)" if status else ""), flush=True)
            time.sleep(2 ** attempt)

def extract_checked(archive, folder):
    folder = Path(folder).resolve()
    folder.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        for info in handle.infolist():
            target = (folder / info.filename).resolve()
            if folder not in target.parents and target != folder:
                raise ValueError("Unsafe archive member: " + info.filename)
        handle.extractall(folder)

def native_mat_paths(folder):
    return [p for p in sorted(Path(folder).rglob("*"))
            if p.is_file() and p.suffix.lower() == ".mat" and p.stem.lower() not in {"cvind", "readme"}]

def fetch_figshare():
    if FIGSHARE_ROOT:
        folder = Path(FIGSHARE_ROOT).expanduser().resolve()
        if not folder.is_dir():
            raise FileNotFoundError(f"FIGSHARE_ROOT does not exist: {folder}. Mount Drive first or correct Cell 1.")
        paths = native_mat_paths(folder)
        if not paths:
            raise FileNotFoundError("No native cjdata .mat files found under FIGSHARE_ROOT: " + str(folder))
        return folder, paths, dict(license="verify supplied local dataset license", local_files=True)
    folder = CACHE / "figshare_1512427_v8"
    folder.mkdir(parents=True, exist_ok=True)
    metadata_path = folder / "publisher_metadata.json"
    try:
        article = get_json("https://api.figshare.com/v2/articles/1512427/versions/8")
        write_json(metadata_path, article)
    except RuntimeError:
        if not metadata_path.is_file():
            raise
        article = json.loads(metadata_path.read_text(encoding="utf-8"))
        print("Publisher API unavailable; using previously saved version-8 metadata and verifying cached MD5s.")
    for entry in article["files"]:
        name = Path(entry["name"]).name
        # Publisher metadata already records provenance. Auxiliary PDF/TXT
        # downloads must not block the required MRI archives.
        if name.lower().endswith(".zip"):
            local = download_url(entry["download_url"], folder / name, entry.get("supplied_md5") or entry.get("computed_md5"))
            if name.lower().endswith(".zip"):
                marker = folder / (name + ".extracted")
                archive_hash = file_sha(local)
                needs_extract = not marker.is_file() or marker.read_text(encoding="utf-8").strip() != archive_hash
                if not needs_extract:
                    with zipfile.ZipFile(local) as handle:
                        needs_extract = any(not (folder / member.filename).is_file() or (folder / member.filename).stat().st_size!=member.file_size
                                            for member in handle.infolist() if member.filename.lower().endswith(".mat"))
                if needs_extract:
                    print("Extracting", name, flush=True)
                    extract_checked(local, folder)
                    marker.write_text(archive_hash,encoding="utf-8")
    paths = native_mat_paths(folder)
    if not paths:
        raise FileNotFoundError("No cjdata MAT files found. Set FIGSHARE_ROOT to the extracted native dataset.")
    return folder, paths, article

### Cell 5 — Read native patients and class folders

Native Figshare reads true `cjdata.PID` values. Folder metadata must contain relative_path,
patient_id and source_dataset; site_id is optional. Never infer patient identity from filenames.

In [ ]:
def decode_patient_id(value):
    if isinstance(value, bytes):
        return value.decode("utf-8").strip()
    if isinstance(value, str):
        return value.strip()
    array = np.asarray(value).ravel()
    if array.dtype.kind in {"U", "S"}:
        return "".join(v.decode("utf-8") if isinstance(v, bytes) else str(v) for v in array).strip()
    if array.size==1 and array.dtype.kind in {"i","u","f"} and np.isfinite(array[0]):
        value=array[0]
        return str(int(value)) if float(value).is_integer() else str(value)
    if array.dtype.kind in {"i", "u", "f"} and len(array) > 1:
        codes = [int(v) for v in array if int(v)]
        if all(0 < v <= 0x10FFFF for v in codes):
            return "".join(chr(v) for v in codes).strip()
    return "".join(str(v) for v in array).strip()

def hdf5_value(handle, dataset):
    value=np.asarray(dataset)
    if h5py.check_dtype(ref=value.dtype) is not None:
        items=[hdf5_value(handle,handle[ref]) for ref in value.ravel() if ref]
        if not items: return np.array([])
        return items[0] if len(items)==1 else np.concatenate([np.asarray(item).ravel() for item in items])
    return value

def read_cheng_mat(path):
    # Original files use MATLAB v7.3/HDF5. Older scipy MAT files are also accepted.
    if h5py.is_hdf5(path):
        with h5py.File(path, "r") as f:
            if "cjdata" not in f:
                raise ValueError("MAT file has no cjdata structure: "+str(path))
            g = f["cjdata"]
            label = int(hdf5_value(f,g["label"]).squeeze())
            pid = decode_patient_id(hdf5_value(f,g["PID"]))
            image = hdf5_value(f,g["image"]).T
            mask = hdf5_value(f,g["tumorMask"]).T.astype(bool) if "tumorMask" in g else None
    else:
        g = scipy.io.loadmat(path, simplify_cells=True)["cjdata"]
        label = int(np.asarray(g["label"]).squeeze())
        raw_pid = g["PID"]
        pid = decode_patient_id(raw_pid)
        image = np.asarray(g["image"])
        mask = np.asarray(g["tumorMask"]).astype(bool) if "tumorMask" in g else None
    if label not in {1, 2, 3} or not pid.strip() or image.ndim != 2 or not image.size or not np.isfinite(image).all():
        raise ValueError("Invalid cjdata label/PID/image: " + str(path))
    if mask is not None and mask.shape != image.shape:
        raise ValueError("Mask and image orientation mismatch: " + str(path))
    return image, mask, pid.strip(), {1: "meningioma", 2: "glioma", 3: "pituitary"}[label]

ALIASES = {"glioma": "glioma", "gliomatumor": "glioma",
    "meningioma": "meningioma", "meningiomatumor": "meningioma",
    "pituitary": "pituitary", "pituitarytumor": "pituitary",
    "notumor": "no_tumor", "notumour": "no_tumor", "normal": "no_tumor"}
def folder_label(path):
    for part in reversed(Path(path).parts[:-1]):
        token = "".join(c for c in part.lower() if c.isalnum())
        if token in ALIASES:
            return ALIASES[token]
    return None
def scan_folder(root, metadata_csv=""):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset folder does not exist: {root}. Mount Drive or correct the root in Cell 1.")
    rows = []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}:
            continue
        label = folder_label(p)
        if label:
            rows.append(dict(path=str(p.resolve()), relative_path=p.relative_to(root).as_posix(),
                label=label, patient_id="", source_dataset="unknown", reader="image", mask_path=""))
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("No class-labelled images found at " + str(root))
    if metadata_csv:
        metadata_path = Path(metadata_csv).expanduser()
        if not metadata_path.is_file():
            raise FileNotFoundError("Metadata CSV not found: " + str(metadata_path))
        meta = pd.read_csv(metadata_path, dtype=str, keep_default_na=False).fillna("")
        required = ["relative_path", "patient_id", "source_dataset"]
        if not set(required).issubset(meta.columns):
            raise ValueError("Metadata requires " + ", ".join(required))
        meta["relative_path"] = meta.relative_path.str.replace("\\", "/", regex=False).str.strip().str.replace(r"^\./", "", regex=True)
        for column in ["patient_id", "source_dataset"]:
            meta[column] = meta[column].str.strip()
        if meta.relative_path.duplicated().any():
            raise ValueError("Duplicate metadata keys.")
        unmatched = set(meta.relative_path) - set(frame.relative_path)
        if unmatched:
            raise ValueError(f"{len(unmatched)} metadata paths do not match images.")
        frame = frame.drop(columns=["patient_id", "source_dataset"]).merge(meta[required + (["site_id"] if "site_id" in meta.columns else [])],
            on="relative_path", how="left", validate="one_to_one").fillna("")
        frame.loc[frame.source_dataset.eq(""), "source_dataset"] = "unknown"
    return frame
def read_image_row(row):
    if row["reader"] == "cheng":
        return read_cheng_mat(row["path"])[0]
    with Image.open(row["path"]) as im:
        return np.asarray(im.convert("L"))
def normalize_image(array, mode="robust"):
    x = np.asarray(array, dtype=np.float32)
    if x.ndim != 2 or not x.size or not np.isfinite(x).all():
        raise ValueError("Image is not a finite 2D array")
    if mode == "robust":
        lo, hi = np.percentile(x, [1, 99])
        x = np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)
    elif mode == "raw":
        # Preserve relative native signal magnitude. This is an acquisition-shortcut diagnostic.
        denominator = 255.0 if np.asarray(array).dtype == np.uint8 else 65535.0
        x = np.clip(x / denominator, 0, 1)
    else:
        raise KeyError(mode)
    return x
def resize_float(x, side=32):
    return np.asarray(Image.fromarray(np.asarray(x, np.float32)).resize(
        (side, side), Image.Resampling.BILINEAR), dtype=np.float32)

### Cell 6 — Acquire a cohort or create an engineering fixture

Synthetic patients are allowed only in the smoke profile. They check code, not efficacy.
Kaggle remains a separate taxonomy and must not be called an independent Figshare replication.

In [ ]:
def make_synthetic():
    folder = CACHE / "synthetic4"
    folder.mkdir(exist_ok=True)
    rng, rows = np.random.default_rng(112), []
    for k, name in enumerate(["glioma", "meningioma", "no_tumor", "pituitary"]):
        (folder / name).mkdir(exist_ok=True)
        for patient in range(36):
            for sl in range(2):
                x = rng.normal(0.15, 0.035, (64, 64))
                yy, xx = np.mgrid[:64, :64]
                brain = ((xx-32)/22)**2 + ((yy-32)/28)**2 < 1
                x[brain] += 0.20
                if name != "no_tumor":
                    center = [(23, 28), (18, 23), (0, 0), (32, 44)][k]
                    x[(xx-center[0])**2+(yy-center[1])**2 < (5+k)**2] += 0.4
                x += patient * 0.0007
                p = folder / name / f"p{patient:03d}_s{sl}.png"
                Image.fromarray((np.clip(x, 0, 1)*255).astype("uint8")).save(p)
                rows.append(dict(path=str(p), relative_path=p.relative_to(folder).as_posix(),
                    label=name, patient_id=f"synthetic_{name}_{patient}",
                    source_dataset="synthetic_A" if patient % 2 else "synthetic_B",
                    reader="image", mask_path=""))
        duplicate = folder / name / "duplicate.png"
        shutil.copy2(folder / name / "p000_s0.png", duplicate)
        original = next(r for r in rows if r["label"] == name)
        rows.append(dict(original, path=str(duplicate), relative_path=duplicate.relative_to(folder).as_posix()))
    return pd.DataFrame(rows), {"synthetic": True, "purpose": "execution check only"}

def acquire_cohort(name):
    if name in {"synthetic3","synthetic4"}:
        frame,provenance=make_synthetic()
        if name=="synthetic3": frame=frame.loc[frame.label.ne("no_tumor")].reset_index(drop=True)
        return frame,dict(provenance,taxonomy=name)
    if name == "figshare3":
        folder, paths, article = fetch_figshare()
        rows = []; discovery_exclusions=[]
        for p in paths:
            try:
                if not h5py.is_hdf5(p):
                    header=scipy.io.whosmat(p)
                    if not any(item[0]=='cjdata' for item in header): continue
                if h5py.is_hdf5(p):
                    with h5py.File(p,'r') as handle:
                        if 'cjdata' not in handle: continue
                image, mask, pid, label = read_cheng_mat(p)
            except (OSError,ValueError,KeyError,TypeError,IndexError,OverflowError,scipy.io.matlab.MatReadError) as exc:
                discovery_exclusions.append(dict(relative_path=p.relative_to(folder).as_posix(),reason='invalid_native_mat',detail=str(exc)))
                continue
            rows.append(dict(path=str(p.resolve()), relative_path=p.relative_to(folder).as_posix(),
                label=label, patient_id="cheng::" + pid, source_dataset="figshare_cheng",
                reader="cheng", mask_path=str(p.resolve()) if mask is not None else ""))
        pd.DataFrame(discovery_exclusions,columns=['relative_path','reason','detail']).to_csv(RUN_ROOT/'figshare_discovery_exclusions.csv',index=False)
        if discovery_exclusions: print('Excluded invalid native MAT files:',len(discovery_exclusions),'See figshare_discovery_exclusions.csv.')
        if not rows: raise ValueError('No valid native cjdata MRI files were found under '+str(folder)+'. See the discovery exclusion report.')
        return pd.DataFrame(rows), dict(discovery_exclusions=len(discovery_exclusions),url="https://doi.org/10.6084/m9.figshare.1512427.v8",
            publisher_version=8, license=article.get("license"), metadata_path=str(folder/"publisher_metadata.json"),
            note="Institution/scanner labels are not reconstructed; source name is not a site identifier.")
    if name == "kaggle4":
        if KAGGLE_ROOT:
            root = Path(KAGGLE_ROOT).expanduser().resolve()
        else:
            import kagglehub
            from kagglehub import clients as kaggle_clients
            # Kagglehub 0.3.13 defaults to a 5-second connection deadline,
            # which is too short on some Colab/proxy connections.
            kaggle_clients.DEFAULT_CONNECT_TIMEOUT = 30
            kaggle_clients.DEFAULT_READ_TIMEOUT = 90
            os.environ.setdefault("KAGGLEHUB_CACHE", str(CACHE / "kagglehub"))
            root = None
            last_error = None
            for attempt in range(3):
                try:
                    root = Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
                    break
                except Exception as exc:
                    last_error = exc
                    if attempt < 2:
                        print(f"Kaggle download retry {attempt + 1}/3: {type(exc).__name__}", flush=True)
                        time.sleep(2 ** attempt)
            if root is None:
                raise RuntimeError("Kaggle download failed. Download/extract the public dataset, then set KAGGLE_ROOT in Cell 1.") from last_error
        return scan_folder(root, METADATA_CSV if METADATA_CSV and not FOLDER_ROOT else ""), dict(handle="masoudnickparvar/brain-tumor-mri-dataset",
            resolved_path=str(root), note="Composite benchmark; patient and original-source metadata unavailable by default.")
    if name == "folder":
        return scan_folder(FOLDER_ROOT, METADATA_CSV), dict(root=FOLDER_ROOT, metadata=METADATA_CSV)
    raise KeyError(name)

### Cell 7 — Audit duplicates, labels and patient connections

Exact copies are removed. High-similarity images are grouped before splitting. Ambiguous
multi-diagnosis groups are quarantined, with an exclusion ledger. Review near-duplicate pairs
manually: image similarity neither proves nor rules out shared patient identity.

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, i):
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.parent[max(a, b)] = min(a, b)
class BKTree:
    def __init__(self):
        self.root = None
    def add(self, value, index):
        if self.root is None:
            self.root = [value, [index], {}]
            return
        node = self.root
        while True:
            distance = (node[0] ^ value).bit_count()
            if distance == 0:
                node[1].append(index)
                return
            if distance not in node[2]:
                node[2][distance] = [value, [index], {}]
                return
            node = node[2][distance]
    def search(self, value, radius):
        stack, found = ([self.root] if self.root else []), []
        while stack:
            node = stack.pop()
            distance = (node[0] ^ value).bit_count()
            if distance <= radius:
                found.extend(node[1])
            stack.extend(child for d, child in node[2].items()
                         if distance-radius <= d <= distance+radius)
        return found

def audit_manifest(frame, folder):
    frame = frame.copy().reset_index(drop=True)
    required = {"path", "relative_path", "label", "patient_id", "source_dataset", "reader", "mask_path"}
    if frame.empty or not required.issubset(frame.columns):
        raise ValueError("Dataset manifest is empty or missing required columns: " + repr(sorted(required - set(frame.columns))))
    for column in ["patient_id", "source_dataset"]:
        frame[column] = frame[column].fillna("").astype(str).str.strip()
    frame.loc[frame.source_dataset.eq(""), "source_dataset"] = "unknown"
    frame=frame.drop(columns=["file_sha","pixel_sha","phash","width","height","valid",
        "image_id","analysis_unit","label_id","partition"],errors="ignore")
    folder.mkdir(parents=True, exist_ok=True)
    fingerprints, small, excluded = [], [], []
    for i, row in frame.iterrows():
        try:
            x = read_image_row(row)
            gray = (resize_float(normalize_image(x), 64)*255).astype("uint8")
            # Exact means native pixels/shape/dtype, not an equality after 32x32 resizing.
            pix = hashlib.sha256(str((x.shape, str(x.dtype))).encode() + x.tobytes()).hexdigest()
            fingerprints.append(dict(file_sha=file_sha(row.path), pixel_sha=pix,
                phash=str(imagehash.phash(Image.fromarray(gray))),
                width=x.shape[1], height=x.shape[0], valid=True))
            small.append(gray)
        except Exception as exc:
            fingerprints.append(dict(file_sha="", pixel_sha="", phash="", width=0, height=0, valid=False))
            small.append(None)
            excluded.append(dict(relative_path=row.relative_path, reason="unreadable", detail=str(exc)))
    frame = pd.concat([frame, pd.DataFrame(fingerprints)], axis=1)
    frame["image_id"] = [digest([r.relative_path, r.file_sha])[:24] for r in frame.itertuples()]
    valid = frame.valid.to_numpy()
    bad_hashes = set(frame.loc[valid].groupby("pixel_sha").label.nunique().loc[lambda s:s>1].index)
    for row in frame.loc[frame.pixel_sha.isin(bad_hashes)].itertuples():
        excluded.append(dict(relative_path=row.relative_path, reason="exact_pixel_label_conflict",
            detail="quarantined; requires source verification; no automatic relabel"))
    valid &= ~frame.pixel_sha.isin(bad_hashes).to_numpy()
    tree, pairs, dsu = BKTree(), [], DSU(len(frame))
    for i in np.where(valid)[0]:
        value = int(frame.iloc[i].phash, 16)
        for j in tree.search(value, PHASH_RADIUS):
            similarity = float(structural_similarity(small[i], small[j], data_range=255))
            same_patient = bool(frame.iloc[i].patient_id and frame.iloc[i].patient_id == frame.iloc[j].patient_id)
            exact = frame.iloc[i].pixel_sha == frame.iloc[j].pixel_sha
            confirmed = exact or similarity >= NEAR_SSIM_THRESHOLD
            pairs.append(dict(image_a=frame.iloc[i].image_id, image_b=frame.iloc[j].image_id,
                path_a=frame.iloc[i].relative_path, path_b=frame.iloc[j].relative_path,
                hamming=(value ^ int(frame.iloc[j].phash,16)).bit_count(), ssim=similarity,
                same_label=frame.iloc[i].label == frame.iloc[j].label,
                same_patient=same_patient, auto_grouped=confirmed,
                note="similarity-based grouping is conservative; not proof of same patient"))
            if confirmed:
                dsu.union(i, j)
        tree.add(value, i)
    # Merge known subject IDs even with partial coverage, and exact duplicates independently of pHash.
    for column in ["patient_id", "pixel_sha"]:
        subset = frame.loc[valid & frame[column].ne("")]
        for _, ids in subset.groupby(column).groups.items():
            ids = list(ids)
            for j in ids[1:]:
                dsu.union(ids[0], j)
    frame["analysis_unit"] = [f"unit_{dsu.find(i):06d}" for i in range(len(frame))]
    frame = frame.loc[valid].copy()
    linked_patient_counts=frame.groupby("analysis_unit").patient_id.apply(lambda s:len(set(s)-{""}))
    # Duplicate images contribute once, with their connected patient links retained.
    redundant = frame.duplicated("pixel_sha", keep="first")
    for row in frame.loc[redundant].itertuples():
        excluded.append(dict(relative_path=row.relative_path, reason="redundant_exact_pixel_copy", detail="first copy retained"))
    frame = frame.loc[~redundant].copy()
    # Homogeneous units are required for mean-probability patient-level multiclass endpoints.
    ambiguous = set(frame.groupby("analysis_unit").label.nunique().loc[lambda s:s>1].index)
    for row in frame.loc[frame.analysis_unit.isin(ambiguous)].itertuples():
        excluded.append(dict(relative_path=row.relative_path, reason="ambiguous_multi_diagnosis_unit",
            detail="excluded from this single-diagnosis endpoint, not declared a mislabeled patient"))
    frame = frame.loc[~frame.analysis_unit.isin(ambiguous)].reset_index(drop=True)
    labels = sorted(frame.label.unique())
    if not 2 <= len(labels) <= 4:
        raise ValueError("The study requires 2–4 retained classes after auditing; found " + repr(labels))
    frame["label_id"] = frame.label.map({name:i for i,name in enumerate(labels)}).astype(int)
    assert (frame.label_id == frame.label.map({name:i for i,name in enumerate(labels)})).all()
    unit_meta = frame.groupby("analysis_unit").agg(label=("label","first"),
        patients=("patient_id", lambda s: len(set(s)-{""})), slices=("image_id","size")).reset_index()
    unit_meta["patients"]=unit_meta.analysis_unit.map(linked_patient_counts)
    patient_coverage = float(frame.patient_id.ne("").mean())
    linked_patients = int((unit_meta.patients > 1).sum())
    audit = dict(discovered=len(valid), retained=len(frame), excluded=len(excluded),
        patient_coverage=patient_coverage, source_coverage=float(frame.source_dataset.ne("unknown").mean()),
        units=len(unit_meta), linked_multi_patient_units=linked_patients,
        complete_patient_unit_claim=patient_coverage==1.0 and linked_patients==0,
        prediction_unit=('patient' if patient_coverage==1.0 and linked_patients==0 else
                         'linked_patient_cluster' if patient_coverage==1.0 else 'unidentified_image_cluster'),
        synthetic=PROFILE=="smoke", exact_conflict_hashes=len(bad_hashes),
        ambiguous_units=len(ambiguous), near_pairs=len(pairs),
        note="Near-duplicate thresholds need manual review and sensitivity analysis for a submission.")
    frame.to_csv(folder/"manifest.csv", index=False)
    pd.DataFrame(excluded, columns=["relative_path","reason","detail"]).to_csv(folder/"exclusions.csv",index=False)
    pd.DataFrame(pairs, columns=["image_a", "image_b", "path_a", "path_b", "hamming", "ssim",
        "same_label", "same_patient", "auto_grouped", "note"]).to_csv(folder/"near_duplicate_review.csv",index=False)
    unit_meta.to_csv(folder/"unit_manifest.csv",index=False)
    write_json(folder/"data_audit.json",audit)
    return frame, labels, audit

### Cell 8 — Build the audited study datasets

The main study requires complete known patient IDs. Exact/near-duplicate links stay together
in every split. If links join different patients with the same diagnosis, the endpoint is
explicitly labelled a **linked patient cluster**, and the individual-patient claim remains
false until source review resolves the links. Missing IDs still block the primary cohort.
Kaggle without verified IDs remains a separate exploratory image-cluster benchmark.


In [ ]:
DATASETS={}
for name in COHORTS:
    raw,provenance=acquire_cohort(name)
    identity=digest([name,[(r.relative_path,file_sha(r.path),r.patient_id,r.source_dataset) for r in raw.itertuples()],PHASH_RADIUS,NEAR_SSIM_THRESHOLD,CODE_SHA])
    folder=RUN_ROOT/name; folder.mkdir(exist_ok=True)
    frame,labels,audit=cached(folder/'audit.joblib',identity,lambda:audit_manifest(raw,folder))
    if name!='kaggle4' and audit['patient_coverage']<1.0:
        raise ValueError('The primary cohort needs complete verified patient IDs. Supply metadata or use native Figshare MAT files. No patient IDs are inferred from filenames.')
    if name!='kaggle4' and audit['linked_multi_patient_units']:
        print('Near/exact duplicates connect',audit['linked_multi_patient_units'],'groups of known patients. Groups remain together in every split; endpoints use linked patient clusters. Review near_duplicate_review.csv before individual-patient claims.')
    frame=frame.reset_index(drop=True)
    if 'site_id' not in frame: frame['site_id']='unknown'
    provenance.update(content_fingerprint=identity,patient_claim=audit['complete_patient_unit_claim'],prediction_unit=audit['prediction_unit'])
    write_json(folder/'provenance.json',provenance)
    ds=dict(name=name,manifest=frame,labels=labels,k=len(labels),audit=audit,folder=folder,
            fingerprint=identity,features={})
    DATASETS[name]=ds
    display(pd.DataFrame([dict(cohort=name,label=label,slices=int((frame.label==label).sum()),
        units=frame.loc[frame.label==label,'analysis_unit'].nunique()) for label in labels]))
    print(name,'patient ID coverage:',audit['patient_coverage'],'retained:',len(frame),'excluded:',audit['excluded'])
if not DATASETS: raise RuntimeError('No dataset loaded.')

### Cell 9 — Write external metadata and review templates

Fill these templates from verified source records. Source names are not scanner identities.
The external cohort must contain the same classes as the primary cohort, have disjoint patients,
and be acquired independently of both original public datasets. Attestations need human verification.

In [ ]:
TEMPLATE_ROOT=RUN_ROOT/'templates'; TEMPLATE_ROOT.mkdir(exist_ok=True)
pd.DataFrame(columns=['relative_path','patient_id','source_dataset','site_id']).to_csv(TEMPLATE_ROOT/'external_metadata.csv',index=False)
write_json(TEMPLATE_ROOT/'external_provenance.json',dict(
 institution='FILL_FROM_SOURCE',dataset_version='FILL_FROM_SOURCE',license='FILL_FROM_SOURCE',
 patient_id_namespace='FILL_FROM_SOURCE',independent_acquisition=False,
 previously_used_for_model_decisions=False,patient_linkage_checked=False,
 public_source_overlap_checked=False,independence_justification='FILL_FROM_SOURCE'))
pd.DataFrame([
 dict(reference='Bowles et al. 2024',existing='Systematic QML versus classical benchmarking',
      ours='Patient-level MRI norm/coherence controls',evidence_needed='Demonstrate a specific additional finding'),
 dict(reference='K.P. et al. 2026',existing='PCA and amplitude-encoded MRI classification',
      ours='Patient splits, matched norm controls, shared learned features',evidence_needed='Reproduce relevant setting fairly; document differences'),
 dict(reference='Angelopoulos and Bates',existing='Split conformal prediction',
      ours='Patient-level empirical reliability comparison',evidence_needed='Calibration size, exchangeability and set efficiency'),
]).to_csv(TEMPLATE_ROOT/'novelty_comparison.csv',index=False)
print('Templates:',TEMPLATE_ROOT)

### Cell 10 — Patient-level split and weighting functions

The calibration floor prevents the previous 6/8-patient full-set fallback. Twelve patients
per class only makes nontrivial thresholds possible; it does not ensure narrow sets or precise
coverage. More independent patients remain valuable. No role shares an analysis unit.

In [ ]:
def units_of(frame):
    return frame[['analysis_unit','label_id']].drop_duplicates().sort_values('analysis_unit').reset_index(drop=True)
def indices_of(frame,units): return frame.index[frame.analysis_unit.isin(set(units))].to_numpy()
def partition_development(frame,development_units,seed):
    u=units_of(frame); u=u.loc[u.analysis_unit.isin(development_units)]
    expected=set(frame.label_id.unique())
    if set(u.label_id.unique())!=expected: raise ValueError('Development patients must include every retained class.')
    rng=np.random.default_rng(seed); roles={x:[] for x in ['fit','selection','temperature','conformal']}
    for label,sub in u.groupby('label_id'):
        ids=sub.analysis_unit.to_numpy().copy(); rng.shuffle(ids); n=len(ids)
        cf=max(CFG['min_cf'],int(np.floor(0.25*n)))
        temp=max(CFG['min_temp'],int(np.floor(0.10*n)))
        sel=max(CFG['min_select'],int(np.floor(0.15*n)))
        fit=n-cf-temp-sel
        if fit<CFG['min_fit']:
            raise ValueError(f'Class {label}: {n} development patients cannot support fit={CFG["min_fit"]}, selection={sel}, temperature={temp}, conformal={cf}. Obtain more patients or redesign prospectively; do not reuse patients.')
        start=0
        for role,count in [('conformal',cf),('temperature',temp),('selection',sel),('fit',fit)]:
            roles[role].extend(ids[start:start+count]); start+=count
    return {role:indices_of(frame,ids) for role,ids in roles.items()}
def assert_split(frame,split):
    seen=set()
    for role,idx in split.items():
        if not len(idx): continue
        ids=set(frame.loc[idx,'analysis_unit'])
        assert not seen.intersection(ids),f'Unit overlap in {role}'
        seen.update(ids)
        assert set(frame.loc[idx,'label_id'])==set(frame.label_id.unique()),f'Missing class in {role}'
    roles=pd.concat([frame.loc[idx,['analysis_unit','patient_id']].assign(role=role) for role,idx in split.items() if len(idx)])
    known=roles[roles.patient_id.ne('')]
    assert known.empty or known.groupby('patient_id').role.nunique().max()==1
def capped_fit(frame,idx,seed):
    rng=np.random.default_rng(seed); result=[]
    for _,sub in frame.loc[idx].groupby('analysis_unit'):
        ii=sub.index.to_numpy(); result.extend(rng.choice(ii,min(len(ii),CFG['slice_cap']),replace=False))
    return np.array(sorted(result),int)
def training_weights(frame,idx,k):
    sub=frame.loc[idx]; w=1/sub.analysis_unit.map(sub.analysis_unit.value_counts()).to_numpy(float)
    for c in range(k): 
        mask=sub.label_id.to_numpy()==c
        if not mask.any(): raise ValueError(f'Fit patients are missing class {c}. Restart with a valid patient split.')
        w[mask]/=w[mask].sum()
    return w/w.mean()
def aggregate_units(frame,idx,p):
    p=np.asarray(p,float)
    if p.ndim!=2 or len(p)!=len(idx) or not np.isfinite(p).all():
        raise ValueError('Probability rows must be finite and match the selected image indices.')
    if not len(idx): raise ValueError('Cannot aggregate an empty evaluation/calibration role.')
    f=frame.loc[idx,['analysis_unit','label_id']].copy()
    for j in range(p.shape[1]): f[f'p{j}']=p[:,j]
    assert f.groupby('analysis_unit').label_id.nunique().max()==1
    return f.groupby('analysis_unit',as_index=False).agg({'label_id':'first',**{f'p{j}':'mean' for j in range(p.shape[1])}})
def unit_probs(table,k): return table[[f'p{j}' for j in range(k)]].to_numpy(float)

### Cell 11 — Freeze repeated outer folds

Each patient is evaluated exactly once per repeat. The same outer and inner partitions are
used by every model and representation. This is repeated held-out evaluation with inner
selection; inner selection itself is a single disjoint holdout, not an inner k-fold average.

In [ ]:
SPLITS={}; split_ledger=[]
for name,ds in DATASETS.items():
    frame=ds['manifest']; u=units_of(frame); plans=[]
    counts=u.label_id.value_counts().sort_index()
    required_dev=CFG['min_cf']+CFG['min_temp']+CFG['min_select']+CFG['min_fit']
    for label,count in counts.items():
        min_dev=int(count)-math.ceil(int(count)/CFG['folds'])
        if int(count)<CFG['folds'] or min_dev<required_dev:
            raise ValueError(f'{name}: class {ds["labels"][label]} has {count} independent units; the smallest outer development fold has {min_dev}, but at least {required_dev} are needed for disjoint fit/selection/temperature/conformal roles. Add verified patients or choose a prospectively smaller profile; calibration patients cannot be reused.')
    for repeat in range(CFG['repeats']):
        cv=StratifiedKFold(CFG['folds'],shuffle=True,random_state=SPLIT_SEED+repeat)
        for fold,(dev,test) in enumerate(cv.split(u,u.label_id)):
            split=partition_development(frame,u.analysis_unit.iloc[dev],SPLIT_SEED+100*repeat+fold+1000)
            split['test']=indices_of(frame,u.analysis_unit.iloc[test]); assert_split(frame,split)
            plan=dict(repeat=repeat,fold=fold,split=split)
            plans.append(plan)
            for role,idx in split.items():
                for c in range(ds['k']):
                    sub=frame.loc[idx]; sub=sub[sub.label_id==c]
                    split_ledger.append(dict(cohort=name,repeat=repeat,fold=fold,role=role,label=ds['labels'][c],
                        patients_or_units=sub.analysis_unit.nunique(),slices=len(sub)))
    SPLITS[name]=plans
pd.DataFrame(split_ledger).to_csv(RUN_ROOT/'split_counts.csv',index=False)
atomic_joblib(SPLITS,RUN_ROOT/'locked_partitions.joblib')
display(pd.DataFrame(split_ledger).query('repeat==0 and fold==0'))

### Cell 12 — Foreground, intensity and perturbation controls

Foreground extraction keeps the largest bright connected component and its bounding box.
It uses no tumor mask or diagnosis. Review the images: this heuristic can remove anatomy.
Perturbations are fixed engineering stress tests applied after intensity normalization;
they are not independent hospitals or scanner acquisitions.

In [ ]:
def foreground_view(x):
    mask=binary_fill_holes(x>0.08)
    labels,count=component_label(mask)
    if not count: return x
    sizes=np.bincount(labels.ravel()); sizes[0]=0; mask=labels==sizes.argmax()
    yy,xx=np.where(mask)
    if len(yy)<0.05*x.size: return x
    return (x*mask)[yy.min():yy.max()+1,xx.min():xx.max()+1]
def perturb(x,kind,seed):
    if kind=='clean': return x
    rng=np.random.default_rng(seed)
    if kind=='noise': out=x+rng.normal(0,0.06,x.shape)
    elif kind=='blur': out=gaussian_filter(x,1.25)
    elif kind=='bias': out=x*np.linspace(0.6,1.4,x.shape[1])[None,:]
    elif kind=='contrast': out=(x-0.5)*0.6+0.5
    else: raise KeyError(kind)
    return np.clip(out,0,1).astype(np.float32)
def prepared_image(row,view,corruption='clean'):
    x=normalize_image(read_image_row(row),'raw' if view=='raw64' else 'robust')
    if view=='foreground64': x=foreground_view(x)
    key=int(hashlib.sha256((str(row['relative_path'])+corruption).encode()).hexdigest()[:8],16)
    return perturb(x,corruption,key)
def border_vector(row):
    x=resize_float(normalize_image(read_image_row(row)),64)
    b=np.r_[x[:6].ravel(),x[-6:].ravel(),x[:,:6].ravel(),x[:,-6:].ravel()]
    return [b.mean(),b.std(),*np.percentile(b,[10,50,90]),row['width']/row['height'],
            np.log1p(row['width']),np.log1p(row['height'])]
fig,axes=plt.subplots(2,3,figsize=(9,6))
first_ds=next(iter(DATASETS.values()))
for col,(_,row) in enumerate(first_ds['manifest'].groupby('label').first().head(3).iterrows()):
    x=normalize_image(read_image_row(row))
    axes[0,col].imshow(x,cmap='gray'); axes[1,col].imshow(foreground_view(x),cmap='gray')
    axes[0,col].set_title('Original normalized'); axes[1,col].set_title('Foreground sensitivity')
for ax in axes.flat: ax.axis('off')
fig.tight_layout(); fig.savefig(RUN_ROOT/'preprocessing_examples.png',dpi=160); plt.show(); plt.close(fig)

### Cell 13 — Shared frozen ResNet18 features

ImageNet weights are required. Feature extraction is label-free and frozen, so it may be
cached across folds. PCA and supervised models are always fitted inside each fold.
Fine-tuned CNNs are trained separately and never supply features to a held-out fold.
Pretrained weights are checksum-checked and retried on download failure. Random weights are never substituted.


In [ ]:
_BACKBONE=None; _BACKBONE_DEVICE=None; BACKBONE_PROVENANCE={}
_RESNET_WEIGHTS=None
def pretrained_resnet18():
    global _RESNET_WEIGHTS
    import torch
    from torchvision.models import resnet18,ResNet18_Weights
    if _RESNET_WEIGHTS is None:
        url=ResNet18_Weights.IMAGENET1K_V1.url
        prefix=Path(urllib.parse.urlsplit(url).path).stem.rsplit('-',1)[-1]
        if RESNET_WEIGHTS_PATH:
            path=Path(RESNET_WEIGHTS_PATH).expanduser()
            if not path.is_file(): raise FileNotFoundError('ResNet weights file does not exist: '+str(path))
        else:
            # Torchvision may already have downloaded these exact weights.
            existing=Path(torch.hub.get_dir())/'checkpoints'/Path(urllib.parse.urlsplit(url).path).name
            path=existing if existing.is_file() and file_sha(existing).startswith(prefix) else CACHE/'weights'/Path(urllib.parse.urlsplit(url).path).name
            if not path.is_file() or not file_sha(path).startswith(prefix):
                if path.exists(): path.rename(path.with_name(path.name+'.invalid.'+str(time.time_ns())))
                path=download_url(url,path,expected_sha256_prefix=prefix)
        if not file_sha(path).startswith(prefix):
            raise ValueError('ResNet18 weights do not match IMAGENET1K_V1. Use the official resnet18-f37072fd.pth file; random weights are not substituted.')
        try: _RESNET_WEIGHTS=torch.load(path,map_location='cpu',weights_only=True)
        except Exception as exc:
            raise RuntimeError('Cannot read the pretrained ResNet weights. Redownload the official file or set QML_RESNET_WEIGHTS_PATH to it.') from exc
    model=resnet18(weights=None)
    model.load_state_dict(_RESNET_WEIGHTS,strict=True)
    return model

def backbone():
    global _BACKBONE,_BACKBONE_DEVICE,BACKBONE_PROVENANCE
    if _BACKBONE is None:
        import torch,torchvision
        from torchvision.models import resnet18,ResNet18_Weights
        _BACKBONE=pretrained_resnet18()
        _BACKBONE.fc=torch.nn.Identity()
        _BACKBONE_DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
        _BACKBONE=_BACKBONE.to(_BACKBONE_DEVICE).eval()
        for p in _BACKBONE.parameters(): p.requires_grad=False
        BACKBONE_PROVENANCE=dict(weights='ResNet18_Weights.IMAGENET1K_V1',
            torch=torch.__version__,torchvision=torchvision.__version__,
            parameters=sum(p.numel() for p in _BACKBONE.parameters()),device=_BACKBONE_DEVICE,weight_source=RESNET_WEIGHTS_PATH or "publisher_sha256_prefix_f37072fd")
        write_json(RUN_ROOT/'backbone_provenance.json',BACKBONE_PROVENANCE)
    return _BACKBONE
def image_tensor(x):
    import torch
    # Same direct 224x224 geometry for frozen and fine-tuned comparisons.
    a=resize_float(x,224)
    t=torch.from_numpy(a.copy()).float().unsqueeze(0).repeat(3,1,1)
    return (t-t.new_tensor([0.485,0.456,0.406])[:,None,None])/t.new_tensor([0.229,0.224,0.225])[:,None,None]
def features_for(frame,view,corruption='clean'):
    if frame.empty: raise ValueError('Cannot extract features from an empty dataset.')
    if view!='resnet18':
        side=32 if view=='pixels32' else 64
        return np.stack([resize_float(prepared_image(row,view,corruption),side).ravel() for _,row in frame.iterrows()])
    import torch
    model=backbone(); outputs=[]
    with torch.inference_mode():
        for start in range(0,len(frame),FEATURE_BATCH_SIZE):
            images=[image_tensor(prepared_image(row,view,corruption)) for _,row in frame.iloc[start:start+FEATURE_BATCH_SIZE].iterrows()]
            x=torch.stack(images).to(_BACKBONE_DEVICE)
            outputs.append(model(x).cpu().numpy())
    return np.concatenate(outputs).astype(np.float32,copy=False)
FEATURE_TIME=[]
for ds in DATASETS.values():
    views=sorted({SPEC_LIBRARY[s]['view'] for s in SPECS})
    for view in views:
        tag=digest([ds['fingerprint'],view,'direct224_imagenet_v1',ENV,CODE_SHA])
        start=time.perf_counter()
        X=cached(CACHE/f'features_{tag[:20]}.joblib',tag,lambda ds=ds,view=view:features_for(ds['manifest'],view))
        assert len(X)==len(ds['manifest']) and np.isfinite(X).all()
        ds['features'][view]=X
        FEATURE_TIME.append(dict(cohort=ds['name'],view=view,seconds_this_session=time.perf_counter()-start,
            dimension=X.shape[1],cache=tag[:20]))
    ds['border']=np.array([border_vector(row) for _,row in ds['manifest'].iterrows()])
pd.DataFrame(FEATURE_TIME).to_csv(RUN_ROOT/'feature_loading_time.csv',index=False)
display(pd.DataFrame(FEATURE_TIME))

### Cell 14 — Fit the representation on training patients only

The norm is the norm of the centered PCA vector, not tumor size or a validated biomarker.
Norm quantiles, PCA and classical scaling never use selection, calibration or test images.
Explained variance and clipped-norm rates are recorded to diagnose compression.

In [ ]:
def represent(X,rep):
    z=np.asarray(rep['pca'].transform(X),dtype=np.float64)
    if z.shape[1]<rep['dimension']: z=np.pad(z,((0,0),(0,rep['dimension']-z.shape[1])))
    norm=np.linalg.norm(z,axis=1)
    d=z/np.maximum(norm[:,None],1e-12)
    zero=norm<1e-12; d[zero]=0; d[zero,0]=1
    lognorm=np.log(np.maximum(norm,1e-12)); lo,hi=rep['norm_limits']
    a=np.pi*np.clip((lognorm-lo)/max(hi-lo,1e-6),0,1)
    return dict(z=z,d=d,a=a,lognorm=lognorm,base=X)
def fit_representation(ds,spec,idx):
    info=SPEC_LIBRARY[spec]; X=ds['features'][info['view']]
    if X.ndim!=2 or not np.isfinite(X).all(): raise ValueError('PCA input must be a finite 2D feature matrix.')
    rank=min(info['dim'],len(idx)-1,X.shape[1])
    if rank<2: raise ValueError('Insufficient training samples for PCA.')
    pca=PCA(n_components=rank,svd_solver='randomized',random_state=SPLIT_SEED).fit(X[idx])
    norms=np.log(np.maximum(np.linalg.norm(pca.transform(X[idx]),axis=1),1e-12))
    limits=np.quantile(norms,[0.05,0.95])
    if limits[1]-limits[0]<1e-6: limits[1]=limits[0]+1
    rep=dict(pca=pca,dimension=info['dim'],spec=spec,view=info['view'],norm_limits=limits,
        rank=rank,explained_variance=float(pca.explained_variance_ratio_.sum()))
    return rep,represent(X,rep)
def quadratic_features(phi):
    i,j=np.triu_indices(phi.shape[1]); return phi[:,i]*phi[:,j]
def classical_features(data,name):
    if name in ['PCA_LR','PCA_SVM','PCA_RF','MLP']: return data['z']
    if name=='Direction_LR': return data['d']
    if name=='DirectionNorm_LR': return np.c_[data['d'],data['a']]
    if name=='NormOnly_LR': return data['a'][:,None]
    if name=='QuadraticDirection_LR': return quadratic_features(data['d'])
    if name=='QuadraticLift_LR': return quadratic_features(np.real(lift_state(data['d'],data['a'],'Passive')))
    if name=='Uncompressed_LR': return data['base']
    raise KeyError(name)

### Cell 15 — Classification metrics and interval helpers

Macro-F1 is primary. Accuracy, balanced accuracy, per-class sensitivity/specificity, AUC,
NLL, Brier score and ECE are secondary. Confidence intervals around coverage are descriptive
binomial intervals for unique held-out patients, not duplicated predictions across repeats.

In [ ]:
def metrics(y, p):
    k = p.shape[1]
    p = np.clip(np.asarray(p,float),1e-12,1); p /= p.sum(1,keepdims=True)
    y = np.asarray(y,int); pred = p.argmax(1)
    cm = confusion_matrix(y,pred,labels=np.arange(k))
    tp, fn, fp = np.diag(cm), cm.sum(1)-np.diag(cm), cm.sum(0)-np.diag(cm)
    tn = cm.sum()-tp-fn-fp
    confidence, correct = p.max(1), (pred==y).astype(float)
    ece = 0.0
    for lo,hi in zip(np.linspace(0,1,11)[:-1],np.linspace(0,1,11)[1:]):
        mask = (confidence>=lo)&(confidence<hi if hi<1 else confidence<=hi)
        if mask.any(): ece += mask.mean()*abs(correct[mask].mean()-confidence[mask].mean())
    auc = np.nan
    if set(np.unique(y))==set(range(k)):
        auc = roc_auc_score(y,p[:,1]) if k==2 else roc_auc_score(y,p,multi_class="ovr",average="macro")
    return dict(accuracy=float((pred==y).mean()), macro_f1=f1_score(y,pred,labels=np.arange(k),average="macro",zero_division=0),
        balanced_accuracy=balanced_accuracy_score(y,pred), macro_auc=auc,
        macro_specificity=float(np.mean(tn/np.maximum(tn+fp,1))),
        nll=log_loss(y,p,labels=np.arange(k)), brier=float(np.mean(((p-np.eye(k)[y])**2).sum(1))),
        ece=float(ece), units=len(y))
def wilson(successes,total,z=1.959963984540054):
    if total==0: return np.nan,np.nan
    p=successes/total; d=1+z*z/total
    center=(p+z*z/(2*total))/d
    radius=z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/d
    return max(0,min(p,center-radius)),min(1,max(p,center+radius))

## Models and mechanistic checks

### Cell 16 — Parameter-matched quantum circuit families

This state-vector implementation uses exact analytic gradients for efficient small-register
training. An independent PennyLane implementation checks it in the next cell. The ancilla
gates are fixed; the trainable data-register rotations are identical across circuit families.

In [ ]:
QUANTUM_MODELS=['Amplitude','Passive','Coherent','ConstantNorm','ShuffledNorm']
def angle_gate(axis,theta):
    if axis=='Z':
        diagonal=np.exp(np.array([-0.5j,0.5j])*theta)
        return np.diag(diagonal),np.diag(diagonal*np.array([-0.5j,0.5j]))
    c,s=np.cos(theta/2),np.sin(theta/2)
    return np.array([[c,-s],[s,c]],complex),0.5*np.array([[-s,-c],[c,-s]],complex)
def apply_gate(state,gate,wire,n):
    return np.einsum('ab,mlbr->mlar',gate,state.reshape(len(state),2**wire,2,2**(n-wire-1)),optimize=True).reshape(state.shape)
def cnot_permutation(n,control,target):
    ids=np.arange(2**n); return ids^(((ids>>(n-1-control))&1)<<(n-1-target))
def circuit_ops(model,weights):
    q=weights.shape[1]; ops=[]
    if model!='Amplitude': ops.append(('CNOT',q,0,None))
    for layer in range(len(weights)):
        for wire in range(q):
            for j,axis in enumerate(['Z','Y','Z']): ops.append((axis,wire,float(weights[layer,wire,j]),(layer,wire,j)))
        for wire in range(q): ops.append(('CNOT',wire,(wire+1)%q,None))
        if model!='Amplitude':
            if model in ['Coherent','ConstantNorm','ShuffledNorm']: ops.append(('H',q,None,None))
            ops.append(('CNOT',q,layer%q,None))
    return ops
def lift_state(d,a,model):
    if model=='Amplitude': return np.asarray(d,complex)
    if model=='ConstantNorm': a=np.full(len(d),np.pi/2)
    anc=np.c_[np.cos(a/2),np.sin(a/2)]
    return (d[:,:,None]*anc[:,None,:]).reshape(len(d),2*d.shape[1]).astype(complex)
def readout_signs(n,k):
    ids=np.arange(2**n); return np.stack([1-2*((ids>>(n-1-wire))&1) for wire in range(k)])
def forward_state(initial,weights,model,save=False):
    n=weights.shape[1]+int(model!='Amplitude'); state=np.asarray(initial,complex); history=[]
    for axis,wire,value,index in circuit_ops(model,weights):
        before=state
        if axis=='CNOT':
            permutation=cnot_permutation(n,wire,value); state=state[:,permutation]
            if save: history.append(('CNOT',permutation,None,None,None))
        else:
            gate,derivative=(np.array([[1,1],[1,-1]],complex)/np.sqrt(2),None) if axis=='H' else angle_gate(axis,value)
            state=apply_gate(state,gate,wire,n)
            if save: history.append((axis,wire,gate,derivative,(index,before)))
    return state,history
def quantum_logits(d,a,weights,model,k):
    state,_=forward_state(lift_state(d,a,model),weights,model)
    return np.abs(state)**2@readout_signs(weights.shape[1]+int(model!='Amplitude'),k).T
def quantum_loss_gradient(d,a,y,weights,model,k,sample_weight):
    state,history=forward_state(lift_state(d,a,model),weights,model,True)
    signs=readout_signs(weights.shape[1]+int(model!='Amplitude'),k)
    logits=np.abs(state)**2@signs.T; p=softmax(logits,axis=1)
    w=np.asarray(sample_weight,float)/len(y)
    loss=np.sum(w*(logsumexp(logits,axis=1)-logits[np.arange(len(y)),y]))
    adjoint=state*(((p-np.eye(k)[y])*w[:,None])@signs); gradient=np.zeros_like(weights)
    for axis,wire,gate,derivative,payload in reversed(history):
        if axis=='CNOT': adjoint=adjoint[:,wire]
        else:
            index,before=payload
            if derivative is not None:
                gradient[index]=2*np.real(np.sum(adjoint.conj()*apply_gate(before,derivative,wire,signs.shape[1].bit_length()-1)))
            adjoint=apply_gate(adjoint,gate.conj().T,wire,signs.shape[1].bit_length()-1)
    return float(loss),gradient
def pennylane_reference(model,d,a,weights,k):
    q=weights.shape[1]; n=q+int(model!='Amplitude')
    @qml.qnode(qml.device('default.qubit',wires=n))
    def circuit():
        qml.AmplitudeEmbedding(d,wires=range(q),normalize=False)
        if model!='Amplitude': qml.RY(np.pi/2 if model=='ConstantNorm' else a,wires=q)
        for axis,wire,value,_ in circuit_ops(model,weights):
            if axis=='CNOT': qml.CNOT(wires=[wire,value])
            elif axis=='H': qml.Hadamard(wires=wire)
            elif axis=='Y': qml.RY(value,wires=wire)
            else: qml.RZ(value,wires=wire)
        return [qml.expval(qml.PauliZ(j)) for j in range(k)]
    return np.asarray(circuit(),float)
def effective_observables(weights,model,k):
    n=weights.shape[1]+int(model!='Amplitude')
    basis,_=forward_state(np.eye(2**n,dtype=complex),weights,model); U=basis.T
    assert np.max(abs(U.conj().T@U-np.eye(len(U))))<1e-9
    return np.stack([np.real(U.conj().T@(s[:,None]*U)) for s in readout_signs(n,k)])
def quadratic_equivalent(d,a,weights,model,k):
    phi=lift_state(d,a,model).real
    return np.einsum('ni,kij,nj->nk',phi,effective_observables(weights,model,k),phi,optimize=True)

### Cell 17 — Independent circuit, gradient and equivalence tests

Checks 4- and 6-data-qubit implementations, norm extremes, every circuit family, numerical
gradients, and the exact passive-mixture identity. The quadratic identity is known mathematics,
not a claimed new theorem. These small models have an exact classical inference realization.

In [ ]:
MATH_CHECKS=[]
for q in [4,6]:
    rng=np.random.default_rng(450+q); d=rng.normal(size=(3,2**q)); d/=np.linalg.norm(d,axis=1,keepdims=True)
    a=np.array([0,0.8,np.pi]); w=rng.normal(0,0.3,(N_LAYERS,q,3)); y=np.arange(3); sw=np.array([0.5,1.,1.5])
    for name in QUANTUM_MODELS:
        native=quantum_logits(d,a,w,name,3)
        reference=np.stack([pennylane_reference(name,di,ai,w,3) for di,ai in zip(d,a)])
        err=float(np.max(abs(native-reference)))
        _,grad=quantum_loss_gradient(d,a,y,w,name,3,sw); errors=[]
        for idx in [(0,0,0),(0,2,1),(1,q-1,2)]:
            plus,minus=w.copy(),w.copy(); plus[idx]+=1e-6; minus[idx]-=1e-6
            fd=(quantum_loss_gradient(d,a,y,plus,name,3,sw)[0]-quantum_loss_gradient(d,a,y,minus,name,3,sw)[0])/2e-6
            errors.append(abs(fd-grad[idx]))
        eq=float(np.max(abs(native-quadratic_equivalent(d,a,w,name,3))))
        assert err<1e-9 and max(errors)<1e-5 and eq<1e-9
        MATH_CHECKS.append(dict(data_qubits=q,model=name,pennylane_error=err,gradient_error=max(errors),quadratic_error=eq))
    passive=quantum_logits(d,a,w,'Passive',3)
    mixture=np.cos(a[:,None]/2)**2*quantum_logits(d,np.zeros(3),w,'Passive',3)+np.sin(a[:,None]/2)**2*quantum_logits(d,np.full(3,np.pi),w,'Passive',3)
    assert np.max(abs(passive-mixture))<1e-9
pd.DataFrame(MATH_CHECKS).to_csv(RUN_ROOT/'implementation_checks.csv',index=False)
display(pd.DataFrame(MATH_CHECKS))

### Cell 18 — Classical models with matched information

All fits use the same patient/class-balanced slice weights as the quantum models.
The MLP is explicitly weighted; it does not silently ignore sample weights. The quadratic
control is independently trained, unlike merely evaluating a trained circuit classically.
The uncompressed linear probe measures the PCA information bottleneck.

In [ ]:
CLASSICAL_MODELS=['PCA_LR','PCA_SVM','PCA_RF','MLP','Direction_LR','DirectionNorm_LR',
    'NormOnly_LR','QuadraticDirection_LR','QuadraticLift_LR','Uncompressed_LR']
def mlp_unpack(theta,d,h,k):
    first=d*h; second=first+h; third=second+h*k
    return theta[:first].reshape(d,h),theta[first:second],theta[second:third].reshape(h,k),theta[third:]
def mlp_logits(model,X):
    A,b,B,c=mlp_unpack(model['theta'],model['d'],model['h'],model['k'])
    return np.tanh(X@A+b)@B+c
def fit_mlp(X,y,k,w,h,l2,seed):
    d=X.shape[1]; rng=np.random.default_rng(seed)
    t=np.r_[rng.normal(0,1/np.sqrt(d),d*h),np.zeros(h),rng.normal(0,1/np.sqrt(h),h*k),np.zeros(k)]
    w=w/w.sum()
    def objective(theta):
        A,b,B,c=mlp_unpack(theta,d,h,k); hidden=np.tanh(X@A+b); logits=hidden@B+c
        p=softmax(logits,axis=1); g=(p-np.eye(k)[y])*w[:,None]; gh=(g@B.T)*(1-hidden**2)
        loss=np.sum(w*(logsumexp(logits,axis=1)-logits[np.arange(len(y)),y]))+l2*np.dot(theta,theta)/2
        grad=np.r_[(X.T@gh).ravel(),gh.sum(0),(hidden.T@g).ravel(),g.sum(0)]+l2*theta
        return loss,grad
    result=minimize(objective,t,jac=True,method='L-BFGS-B',options={'maxiter':30 if PROFILE=='smoke' else 500})
    return dict(theta=result.x,d=d,h=h,k=k,converged=bool(result.success),message=str(result.message))
def classical_grid(name):
    if name=='PCA_RF': return [dict(max_depth=d,min_samples_leaf=2) for d in ([8,None] if PROFILE=='paper' else [8])]
    if name=='PCA_SVM': return [dict(C=c,gamma=g) for c in CFG['Cs'] for g in (['scale',0.03] if PROFILE=='paper' else ['scale'])]
    if name=='MLP': return [dict(h=h,l2=l2) for h in ([4,16] if PROFILE=='paper' else [4]) for l2 in ([0.001,0.01] if PROFILE=='paper' else [0.01])]
    return [dict(C=c) for c in CFG['Cs']]
def fit_classical(name,parameters,X,y,k,w,seed):
    scaler=StandardScaler().fit(X); X=scaler.transform(X)
    if name=='MLP':
        est=fit_mlp(X,y,k,w,seed=seed,**parameters); count=len(est['theta']); converged=est['converged']
    else:
        if name=='PCA_RF': est=RandomForestClassifier(n_estimators=40 if PROFILE=='smoke' else 200,n_jobs=2,random_state=seed,**parameters)
        elif name=='PCA_SVM': est=SVC(probability=False,decision_function_shape='ovr',**parameters)
        else: est=LogisticRegression(max_iter=3000,random_state=seed,**parameters)
        est.fit(X,y,sample_weight=w)
        count=int(est.coef_.size+est.intercept_.size) if hasattr(est,'coef_') else None
        converged=not hasattr(est,'n_iter_') or int(np.max(est.n_iter_))<3000
    return dict(name=name,parameters=parameters,scaler=scaler,estimator=est,trainable=count,converged=converged)
def classical_logits(record,X):
    X=record['scaler'].transform(X); est=record['estimator']
    if record['name']=='MLP': return mlp_logits(est,X)
    if isinstance(est,RandomForestClassifier): return np.log(np.clip(est.predict_proba(X),1e-12,1))
    logits=est.decision_function(X)
    return np.c_[-logits/2,logits/2] if logits.ndim==1 else np.asarray(logits,float)

### Cell 19 — Quantum training and convergence logs

Adam optimizes weighted cross-entropy; selection macro-F1, then NLL, selects the checkpoint.
Warmup prevents premature stopping. Learning-rate candidates and training budgets are identical
across the quantum families. Increasing capacity or trying new optimizers after seeing outer
results constitutes further exploratory development.

In [ ]:
def angles_for(data,name,seed,role):
    a=data['a'].copy()
    if name=='ShuffledNorm':
        salt=int(hashlib.sha256(role.encode()).hexdigest()[:8],16)
        a=np.random.default_rng(seed+salt).permutation(a)
    return a
def subset_data(data,idx): return {key:value[idx] for key,value in data.items()}
def train_quantum(name,lr,data,ds,fit_idx,selection_idx,seed):
    frame,k=ds['manifest'],ds['k']; q=int(np.log2(data['d'].shape[1])); rng=np.random.default_rng(seed)
    weights=rng.normal(0,0.08,(N_LAYERS,q,3)); first=np.zeros_like(weights); second=first.copy()
    y=frame.label_id.to_numpy(); sw=training_weights(frame,fit_idx,k)
    fit=subset_data(data,fit_idx); val=subset_data(data,selection_idx)
    a_fit=angles_for(fit,name,seed,'fit'); a_val=angles_for(val,name,seed,'selection')
    best=None; stale=0; step=0; history=[]
    for epoch in range(1,CFG['epochs']+1):
        order=rng.permutation(len(fit_idx)); losses=[]; grad_norms=[]
        for begin in range(0,len(order),CFG['batch']):
            b=order[begin:begin+CFG['batch']]
            loss,g=quantum_loss_gradient(fit['d'][b],a_fit[b],y[fit_idx[b]],weights,name,k,sw[b])
            assert np.isfinite(loss) and np.isfinite(g).all()
            gn=float(np.linalg.norm(g)); grad_norms.append(gn); g*=min(1,5/max(gn,1e-12))
            step+=1; first=0.9*first+0.1*g; second=0.999*second+0.001*g*g
            weights-=lr*(first/(1-0.9**step))/(np.sqrt(second/(1-0.999**step))+1e-8)
            losses.append((loss,len(b)))
        p=softmax(quantum_logits(val['d'],a_val,weights,name,k),axis=1)
        table=aggregate_units(frame,selection_idx,p); vals=metrics(table.label_id,unit_probs(table,k))
        score=(vals['macro_f1'],-vals['nll'])
        history.append(dict(epoch=epoch,train_loss=sum(l*n for l,n in losses)/sum(n for _,n in losses),
            selection_macro_f1=score[0],selection_nll=-score[1],gradient_norm=np.mean(grad_norms)))
        if best is None or score>best['score']: best=dict(score=score,weights=weights.copy(),epoch=epoch); stale=0
        else: stale+=1
        if epoch>=CFG['warmup'] and stale>=CFG['patience']: break
    return dict(name=name,weights=best['weights'],score=best['score'],seed=seed,lr=lr,
        parameters={'lr':lr},trainable=weights.size,selected_epoch=best['epoch'],
        history=pd.DataFrame(history),budget_exhausted=epoch==CFG['epochs'])

### Cell 20 — Fine-tuned ResNet18 practical comparator

The paper profile includes a full fine-tune, with pretrained weights, training-only
augmentation, patient/class weighting, fixed AdamW rates and selection-only early stopping.
It runs once per outer split with seed 13 and is reused across quantum initialization seeds.
This avoids falsely counting repeated copies as independent CNN fits. It uses native images
resized to 224 pixels and is a practical comparator, not a parameter-matched quantum head.
Training and inference load only the current image batch to limit RAM use.


In [ ]:
def cnn_logits_state(record,frame,idx):
    import torch
    from torchvision.models import resnet18
    if not len(idx): return np.empty((0,record['k']),dtype=np.float32)
    model=resnet18(weights=None); model.fc=torch.nn.Linear(model.fc.in_features,record['k'])
    model.load_state_dict(record['state']); device='cuda' if torch.cuda.is_available() else 'cpu'
    model=model.to(device).eval(); outputs=[]
    with torch.inference_mode():
        for start in range(0,len(idx),CNN_INFERENCE_BATCH_SIZE):
            xx=torch.stack([image_tensor(prepared_image(frame.loc[i],'resnet18')) for i in idx[start:start+CNN_INFERENCE_BATCH_SIZE]]).to(device)
            outputs.append(model(xx).cpu().numpy())
    del model
    if device=='cuda': torch.cuda.empty_cache()
    return np.concatenate(outputs)
def train_cnn(ds,fit_idx,selection_idx):
    import torch
    import torch.nn.functional as F
    from torchvision.models import resnet18,ResNet18_Weights
    from torchvision.transforms import functional as TF
    seed_everything(13); frame=ds['manifest']; k=ds['k']
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model=pretrained_resnet18()
    model.fc=torch.nn.Linear(model.fc.in_features,k); model=model.to(device)
    optimizer=torch.optim.AdamW([
        {'params':[p for n,p in model.named_parameters() if not n.startswith('fc.')],'lr':1e-5},
        {'params':model.fc.parameters(),'lr':1e-3}],weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG['cnn_epochs'])
    # Load only the current batch; preloading every 224x224 RGB tensor can exhaust Colab RAM.
    if not len(fit_idx) or not len(selection_idx): raise ValueError('CNN needs nonempty fit and selection roles.')
    y=frame.label_id.to_numpy(); sw=training_weights(frame,fit_idx,k)
    best=None; stale=0; history=[]; rng=np.random.default_rng(13)
    for epoch in range(1,CFG['cnn_epochs']+1):
        model.train(); losses=[]; order=rng.permutation(len(fit_idx))
        for start in range(0,len(order),CNN_BATCH_SIZE):
            b=order[start:start+CNN_BATCH_SIZE]; batch=[]
            for i in b:
                x=image_tensor(prepared_image(frame.loc[fit_idx[i]],'resnet18'))
                if rng.random()<0.5: x=TF.hflip(x)
                x=TF.rotate(x,float(rng.uniform(-10,10))); batch.append(x)
            xx=torch.stack(batch).to(device); yy=torch.as_tensor(y[fit_idx[b]],device=device)
            ww=torch.as_tensor(sw[b],device=device,dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True); loss=(F.cross_entropy(model(xx),yy,reduction='none')*ww).mean()
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5); optimizer.step()
            losses.append((float(loss.detach()),len(b)))
        model.eval(); pred=[]
        with torch.inference_mode():
            for start in range(0,len(selection_idx),CNN_INFERENCE_BATCH_SIZE):
                ids=selection_idx[start:start+CNN_INFERENCE_BATCH_SIZE]
                xx=torch.stack([image_tensor(prepared_image(frame.loc[i],'resnet18')) for i in ids]).to(device)
                pred.append(model(xx).cpu().numpy())
        tab=aggregate_units(frame,selection_idx,softmax(np.concatenate(pred),axis=1)); met=metrics(tab.label_id,unit_probs(tab,k))
        score=(met['macro_f1'],-met['nll'])
        history.append(dict(epoch=epoch,train_loss=sum(l*n for l,n in losses)/sum(n for _,n in losses),selection_macro_f1=score[0],selection_nll=-score[1]))
        if best is None or score>best['score']:
            best=dict(score=score,state={n:p.detach().cpu().clone() for n,p in model.state_dict().items()},epoch=epoch); stale=0
        else: stale+=1
        scheduler.step()
        if epoch>=5 and stale>=7: break
    record=dict(name='FineTunedResNet18',k=k,state=best['state'],score=best['score'],seed=13,
        selected_epoch=best['epoch'],trainable=sum(p.numel() for p in model.parameters()),history=pd.DataFrame(history),
        parameters={'backbone_lr':1e-5,'head_lr':1e-3,'weight_decay':1e-4},budget_exhausted=epoch==CFG['cnn_epochs'])
    del model,optimizer,scheduler
    if device=='cuda': torch.cuda.empty_cache()
    return record

### Cell 21 — Separate temperature and class-conditional set calibration

LAC is the primary score (1 minus class probability). APS uses cumulative probability in
descending class order and is a prespecified secondary score. Report coverage together with
set size, full-set rate and singleton rate. A set containing every class is flagged as
uninformative even when its coverage is perfect.

In [ ]:
def temperature_fit(frame,idx,logits,k):
    def objective(logt):
        tab=aggregate_units(frame,idx,softmax(logits/np.exp(logt),axis=1))
        return log_loss(tab.label_id,unit_probs(tab,k),labels=np.arange(k))
    fit=minimize_scalar(objective,bounds=(-4,4),method='bounded')
    return float(np.exp(fit.x))
def score_matrix(p,method):
    if method=='LAC': return 1-p
    order=np.argsort(-p,axis=1,kind='stable'); cumulative=np.cumsum(np.take_along_axis(p,order,axis=1),axis=1)
    out=np.empty_like(p); np.put_along_axis(out,order,cumulative,axis=1); return np.clip(out,0,1)
def conformal_fit(y,p,method):
    scores=score_matrix(p,method); thresholds=[]; counts=[]; fallback=[]
    for j in range(p.shape[1]):
        v=scores[y==j,j]; n=len(v); rank=int(np.ceil((n+1)*(1-ALPHA)))
        thresholds.append(1.0 if rank>n else float(np.partition(v,rank-1)[rank-1]))
        counts.append(n); fallback.append(rank>n)
    return dict(method=method,thresholds=np.array(thresholds),counts=counts,fallback=fallback)
def conformal_sets(p,record): return score_matrix(p,record['method'])<=record['thresholds'][None,:]+1e-12
def set_metrics(y,sets):
    size=sets.sum(1); hits=sets[np.arange(len(y)),np.asarray(y,int)]; single=size==1
    lo,hi=wilson(int(hits.sum()),len(y))
    return dict(coverage=float(hits.mean()),coverage_ci_low=lo,coverage_ci_high=hi,mean_set_size=float(size.mean()),
        full_set_rate=float((size==sets.shape[1]).mean()),singleton_rate=float(single.mean()),
        singleton_accuracy=float(hits[single].mean()) if single.any() else np.nan,empty_rate=float((size==0).mean()))
# The previous 8/8/6 calibration configuration must be detected as a full-set fallback.
toy_p=np.full((22,3),1/3); toy_y=np.repeat(np.arange(3),[8,8,6])
assert all(conformal_fit(toy_y,toy_p,'LAC')['fallback'])
assert conformal_sets(np.array([[.9,.05,.05]]),conformal_fit(toy_y,toy_p,'LAC')).all()
print('Small-calibration guard checked. Real folds reserve at least',CFG['min_cf'],'units per class.')

### Cell 22 — Checkpointed fitting and model selection

Candidates are selected before calibration and before outer evaluation. Cache identities
include code, software, dataset, split, representation and hyperparameters. Corrupt caches are
quarantined. Fine-tuned CNN fitting is shared only within the same outer split, never across
folds. Secondary representations retain the same quantum controls and a reduced classical
suite; the full classical suite is used for the primary representation.

In [ ]:
def slice_logits(record,data,frame,idx,role):
    if record['name']=='FineTunedResNet18': return cnn_logits_state(record,frame,idx)
    part=subset_data(data,idx)
    if record['name'] in QUANTUM_MODELS:
        a=angles_for(part,record['name'],record['seed'],role)
        return quantum_logits(part['d'],a,record['weights'],record['name'],len(frame.label_id.unique()))
    return classical_logits(record,classical_features(part,record['name']))
def fit_bundle(ds,spec,plan,seed,stage='internal'):
    frame,k=ds['manifest'],ds['k']; split=plan['split']; assert_split(frame,split)
    fit_idx=capped_fit(frame,split['fit'],SPLIT_SEED+plan['repeat']*100+plan['fold'])
    identity=digest([PROTOCOL_HASH,RUNTIME_FUNCTION_HASH,ds['fingerprint'],spec,split,fit_idx,seed,stage])
    folder=ds['folder']/stage/f'r{plan["repeat"]}_f{plan["fold"]}'/spec/f'seed{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    def build():
        rep,data=fit_representation(ds,spec,fit_idx)
        selected={}; ledger=[]
        names=CLASSICAL_MODELS if spec==PRIMARY_SPEC else ['PCA_LR','PCA_SVM','DirectionNorm_LR','QuadraticLift_LR','Uncompressed_LR']
        for name in names+QUANTUM_MODELS:
            print('  Fitting',name,'on',len(fit_idx),'fit slices',flush=True)
            grid=[dict(lr=lr) for lr in CFG['lrs']] if name in QUANTUM_MODELS else classical_grid(name)
            candidates=[]
            for parameters in grid:
                tag=digest([identity,name,parameters])
                def fit_candidate():
                    start=time.perf_counter()
                    if name in QUANTUM_MODELS:
                        rec=train_quantum(name,parameters['lr'],data,ds,fit_idx,split['selection'],seed)
                    else:
                        X=classical_features(data,name)
                        rec=fit_classical(name,parameters,X[fit_idx],frame.label_id.to_numpy()[fit_idx],k,training_weights(frame,fit_idx,k),seed)
                        logits=classical_logits(rec,X[split['selection']]); tab=aggregate_units(frame,split['selection'],softmax(logits,axis=1))
                        met=metrics(tab.label_id,unit_probs(tab,k)); rec['score']=(met['macro_f1'],-met['nll'])
                    rec['fit_seconds']=time.perf_counter()-start
                    return rec
                rec=cached(folder/f'candidate_{tag[:16]}.joblib',tag,fit_candidate)
                candidates.append(rec)
                ledger.append(dict(model=name,seed=seed,parameters=json.dumps(parameters),selection_macro_f1=rec['score'][0],
                    selection_nll=-rec['score'][1],fit_seconds=rec['fit_seconds'],selected=False))
            best_index=max(range(len(candidates)),key=lambda i:candidates[i]['score'])
            best=copy.deepcopy(candidates[best_index]); best['search_seconds']=sum(r['fit_seconds'] for r in candidates)
            ledger[-len(candidates)+best_index]['selected']=True; selected[name]=best
            if 'history' in best: best['history'].to_csv(folder/f'history_{name}.csv',index=False)
        if RUN_FINE_TUNE and spec==PRIMARY_SPEC:
            tag=digest([PROTOCOL_HASH,RUNTIME_FUNCTION_HASH,ds['fingerprint'],split,fit_idx,'cnn_seed13',stage])
            cnn_path=ds['folder']/stage/f'cnn_{tag[:16]}.joblib'
            def fit_cnn():
                start=time.perf_counter(); rec=train_cnn(ds,fit_idx,split['selection']); rec['fit_seconds']=time.perf_counter()-start
                rec['search_seconds']=rec['fit_seconds']; return rec
            selected['FineTunedResNet18']=cached(cnn_path,tag,fit_cnn)
            selected['FineTunedResNet18']['history'].to_csv(folder/'history_FineTunedResNet18.csv',index=False)
            names=names+['FineTunedResNet18']
        chosen=max(names,key=lambda name:selected[name]['score'])
        for name,rec in selected.items():
            ti=split['temperature']; ci=split['conformal']
            rec['temperature']=temperature_fit(frame,ti,slice_logits(rec,data,frame,ti,'temperature'),k)
            tab=aggregate_units(frame,ci,softmax(slice_logits(rec,data,frame,ci,'conformal')/rec['temperature'],axis=1))
            rec['conformal']={method:conformal_fit(tab.label_id.to_numpy(),unit_probs(tab,k),method) for method in ['LAC','APS']}
            assert not any(rec['conformal']['LAC']['fallback']),'Calibration role too small.'
        pd.DataFrame(ledger).to_csv(folder/'search_ledger.csv',index=False)
        return dict(identity=identity,spec=spec,rep=rep,models=selected,selected_classical=chosen,
            split=split,fit_idx=fit_idx,repeat=plan['repeat'],fold=plan['fold'],seed=seed,
            cohort=ds['name'],folder=str(folder),stage=stage)
    return cached(folder/'bundle.joblib',identity,build)
import types
def code_fingerprint(obj):
    if isinstance(obj,types.CodeType):
        return [obj.co_code.hex(),obj.co_names,obj.co_varnames,[code_fingerprint(c) for c in obj.co_consts]]
    if isinstance(obj,tuple): return [code_fingerprint(x) for x in obj]
    return obj if isinstance(obj,(str,int,float,bool,type(None))) else str(obj)
RUNTIME_FUNCTION_HASH=digest([code_fingerprint(f.__code__) for f in [
    normalize_image,foreground_view,prepared_image,pretrained_resnet18,features_for,represent,fit_representation,
    partition_development,capped_fit,training_weights,aggregate_units,metrics,
    angle_gate,apply_gate,circuit_ops,lift_state,forward_state,quantum_loss_gradient,
    train_quantum,fit_classical,classical_features,classical_logits,fit_mlp,train_cnn,
    temperature_fit,score_matrix,conformal_fit,slice_logits,fit_bundle]])
write_json(RUN_ROOT/'runtime_implementation_hash.json',{'sha256':RUNTIME_FUNCTION_HASH})

### Cell 23 — Evaluate fixed models and reconcile every output

Test probabilities are averaged over slices before computing patient metrics. Per-class
metrics and both conformal methods are retained. The validation-selected classical strategy
is copied only after it has been selected on inner patients; no outer best-model selection occurs.

In [ ]:
def evaluate_bundle(ds,bundle,idx,role='test',data_override=None):
    frame,k=ds['manifest'],ds['k']; rep=bundle['rep']
    data=represent(ds['features'][rep['view']],rep) if data_override is None else data_override
    prediction_tables=[]; metric_rows=[]; class_rows=[]; reliability_rows=[]
    for name,rec in bundle['models'].items():
        start=time.perf_counter(); logits=slice_logits(rec,data,frame,idx,role); infer=time.perf_counter()-start
        tab=aggregate_units(frame,idx,softmax(logits/rec['temperature'],axis=1)); p=unit_probs(tab,k); y=tab.label_id.to_numpy()
        info=dict(cohort=ds['name'],spec=bundle['spec'],repeat=bundle['repeat'],fold=bundle['fold'],seed=bundle['seed'],model=name)
        met=metrics(y,p); row=dict(**info,**met,inference_seconds=infer,selection_macro_f1=rec['score'][0],
            trainable_parameters=rec['trainable'],fit_seconds=rec['fit_seconds'],search_seconds=rec['search_seconds'],temperature=rec['temperature'])
        for method,cf in rec['conformal'].items():
            sets=conformal_sets(p,cf)
            reliability_rows.append(dict(**info,method=method,**set_metrics(y,sets),test_units=len(y),
                min_calibration_units=min(cf['counts']),finite_thresholds=not any(cf['fallback']),
                patient_independence_verified=ds['audit']['patient_coverage']==1.0,prediction_unit=ds['audit']['prediction_unit']))
            for j in range(k): tab[f'{method}_set{j}']=sets[:,j].astype(int)
        cm=confusion_matrix(y,p.argmax(1),labels=np.arange(k))
        precision,recall,f1,support=precision_recall_fscore_support(y,p.argmax(1),labels=np.arange(k),zero_division=0)
        for j in range(k):
            tp=cm[j,j]; fn=cm[j].sum()-tp; fp=cm[:,j].sum()-tp; tn=cm.sum()-tp-fn-fp
            class_rows.append(dict(**info,label=ds['labels'][j],precision=precision[j],sensitivity=recall[j],f1=f1[j],
                specificity=tn/max(tn+fp,1),support=int(support[j])))
        for key,value in info.items(): tab[key]=value
        tab['prediction']=p.argmax(1); prediction_tables.append(tab); metric_rows.append(row)
    chosen=bundle['selected_classical']
    prediction_tables.append(next(t for t in prediction_tables if t.model.iloc[0]==chosen).assign(model='SelectedClassical'))
    metric_rows.append(dict(next(r for r in metric_rows if r['model']==chosen),model='SelectedClassical',selected_family=chosen))
    reliability_rows.extend([dict(r,model='SelectedClassical') for r in reliability_rows if r['model']==chosen])
    class_rows.extend([dict(r,model='SelectedClassical') for r in class_rows if r['model']==chosen])
    return dict(predictions=pd.concat(prediction_tables,ignore_index=True),metrics=pd.DataFrame(metric_rows),
        classes=pd.DataFrame(class_rows),reliability=pd.DataFrame(reliability_rows))

### Cell 24 — Execute the complete repeated study

This is the main long-running cell. A paper run performs 10 outer splits, seven
representations, and three primary initialization seeds. Secondary representations use one
seed per split. Completed fits and evaluations resume automatically. Do not use pilot or smoke
numbers as publication results. Runtime depends on GPU, patients, optimization and cache reuse.

In [ ]:
TASKS=[]
for name,ds in DATASETS.items():
    for plan in SPLITS[name]:
        for spec in SPECS:
            seeds=CFG['seeds'] if spec==PRIMARY_SPEC else [CFG['seeds'][0]]
            for seed in seeds: TASKS.append((name,plan,spec,seed))
print('Fold/representation/seed tasks:',len(TASKS),'| fine-tuned CNN fits:',sum(len(SPLITS[n]) for n in DATASETS) if RUN_FINE_TUNE else 0)
BUNDLE_INDEX=[]; outputs=[]
for task_number,(name,plan,spec,seed) in enumerate(TASKS,1):
    ds=DATASETS[name]
    print(f'[{task_number}/{len(TASKS)}] {name} repeat {plan["repeat"]} fold {plan["fold"]} {spec} seed {seed}',flush=True)
    bundle=fit_bundle(ds,spec,plan,seed)
    tag=digest([bundle['identity'],'outer_evaluation'])
    result=cached(Path(bundle['folder'])/'evaluation.joblib',tag,lambda:evaluate_bundle(ds,bundle,plan['split']['test']))
    for table,value in result.items(): value.to_csv(Path(bundle['folder'])/f'{table}.csv',index=False)
    BUNDLE_INDEX.append(dict(cohort=name,spec=spec,repeat=plan['repeat'],fold=plan['fold'],seed=seed,path=str(Path(bundle['folder'])/'bundle.joblib')))
    outputs.append(result)
    del bundle
PREDICTIONS=pd.concat([r['predictions'] for r in outputs],ignore_index=True)
FOLD_METRICS=pd.concat([r['metrics'] for r in outputs],ignore_index=True)
CLASS_METRICS=pd.concat([r['classes'] for r in outputs],ignore_index=True)
RELIABILITY=pd.concat([r['reliability'] for r in outputs],ignore_index=True)
pd.DataFrame(BUNDLE_INDEX).to_csv(RUN_ROOT/'bundle_index.csv',index=False)
for name,df in [('outer_predictions',PREDICTIONS),('fold_metrics',FOLD_METRICS),('class_metrics',CLASS_METRICS),('reliability',RELIABILITY)]: df.to_csv(RUN_ROOT/f'{name}.csv',index=False)
del outputs
print('All completed predictions saved. Model choices were made before each fold evaluation.')

## Results — generated from this run only

### Cell 25 — One prediction per patient per repeat

Pool the disjoint outer predictions within each repeat before computing macro-F1.
Report variability over repeat/seed runs descriptively; do not inflate the sample size by
counting multiple predictions of the same patient as new patients.

In [ ]:
OOF_ROWS=[]
group_keys=['cohort','spec','repeat','seed','model']
for key,tab in PREDICTIONS.groupby(group_keys):
    cohort,spec,repeat,seed,model=key; ds=DATASETS[cohort]
    assert not tab.analysis_unit.duplicated().any(),'Duplicate out-of-fold patient'
    assert set(tab.analysis_unit)==set(ds['manifest'].analysis_unit),'Incomplete outer coverage'
    OOF_ROWS.append(dict(zip(group_keys,key),**metrics(tab.label_id,unit_probs(tab,ds['k']))))
OOF=pd.DataFrame(OOF_ROWS)
SUMMARY=OOF.groupby(['cohort','spec','model']).agg(macro_f1=('macro_f1','mean'),
    repeat_seed_sd=('macro_f1','std'),accuracy=('accuracy','mean'),balanced_accuracy=('balanced_accuracy','mean'),
    nll=('nll','mean'),brier=('brier','mean'),ece=('ece','mean'),unique_units=('units','first'),runs=('seed','size')).reset_index()
OOF.to_csv(RUN_ROOT/'oof_run_metrics.csv',index=False); SUMMARY.to_csv(RUN_ROOT/'model_summary.csv',index=False)
display(SUMMARY.loc[SUMMARY.spec==PRIMARY_SPEC].round(4))

### Cell 26 — Paired patient effect estimates without false independence

Patients, not slices, are resampled. The same patient draw is reused across repeat/seed
prediction arrays, preserving pairing. These internal bootstrap intervals condition on OOF
predictions and ignore training-set overlap/refitting uncertainty: they are exploratory
descriptions, not confirmatory tests. No internal p-values are calculated.

In [ ]:
def aligned_oof(cohort,spec,model):
    tabs=[]; reference=None
    sub=PREDICTIONS.query('cohort==@cohort and spec==@spec and model==@model')
    for _,tab in sub.groupby(['repeat','seed'],sort=True):
        tab=tab.sort_values('analysis_unit'); identity=tab[['analysis_unit','label_id']].reset_index(drop=True)
        if reference is None: reference=identity
        else: pd.testing.assert_frame_equal(reference,identity)
        tabs.append(unit_probs(tab,DATASETS[cohort]['k']))
    return reference,np.stack(tabs)
def mean_f1(y,prob):
    k=prob.shape[-1]; values=[]
    for p in prob:
        cm=np.bincount(k*np.asarray(y,int)+p.argmax(1),minlength=k*k).reshape(k,k)
        den=cm.sum(0)+cm.sum(1)
        values.append(np.divide(2*np.diag(cm),den,out=np.zeros(k,float),where=den>0).mean())
    return float(np.mean(values))
def paired_interval(y,a,b,repetitions,seed=712):
    rng=np.random.default_rng(seed); strata=[np.where(y==j)[0] for j in range(a.shape[-1])]
    draws=[]
    for _ in range(repetitions):
        idx=np.concatenate([rng.choice(s,len(s),replace=True) for s in strata])
        draws.append(mean_f1(y[idx],a[:,idx])-mean_f1(y[idx],b[:,idx]))
    return dict(delta=mean_f1(y,a)-mean_f1(y,b),ci_low=float(np.quantile(draws,.025)),ci_high=float(np.quantile(draws,.975)))
EFFECT_ROWS=[]
for cohort in DATASETS:
    for spec in SPECS:
        ref,a=aligned_oof(cohort,spec,'Coherent')
        for comparator in ['Passive','Amplitude','SelectedClassical','ConstantNorm','ShuffledNorm','QuadraticLift_LR']:
            ref2,b=aligned_oof(cohort,spec,comparator); pd.testing.assert_frame_equal(ref,ref2)
            EFFECT_ROWS.append(dict(cohort=cohort,spec=spec,comparator=comparator,
                **paired_interval(ref.label_id.to_numpy(),a,b,CFG['bootstrap']),unique_units=len(ref),
                interpretation='exploratory; conditional on dependent OOF predictions; not a confirmation test'))
EFFECTS=pd.DataFrame(EFFECT_ROWS); EFFECTS.to_csv(RUN_ROOT/'exploratory_paired_effects.csv',index=False)
display(EFFECTS.loc[EFFECTS.spec==PRIMARY_SPEC].round(4))

### Cell 27 — Calibration efficiency and class-wise coverage

Per-class coverage is computed within each repeat/seed after collecting each patient's
one held-out prediction. Repeats are never pooled into a larger binomial denominator.
Pooled OOF Wilson intervals are descriptive: they do not account for dependence induced
by shared training/calibration sets across folds.
Near-perfect coverage with large full-set rates must be described as low efficiency.

In [ ]:
CLASS_COVERAGE=[]; SET_RUNS=[]
for key,tab in PREDICTIONS.groupby(group_keys):
    info=dict(zip(group_keys,key)); k=DATASETS[info['cohort']]['k']; y=tab.label_id.to_numpy()
    for method in ['LAC','APS']:
        sets=tab[[f'{method}_set{j}' for j in range(k)]].to_numpy(bool)
        SET_RUNS.append(dict(**info,method=method,**set_metrics(y,sets)))
        for j,label in enumerate(DATASETS[info['cohort']]['labels']):
            hit=sets[y==j,j]; lo,hi=wilson(int(hit.sum()),len(hit))
            CLASS_COVERAGE.append(dict(**info,method=method,label=label,coverage=hit.mean(),ci_low=lo,ci_high=hi,units=len(hit)))
CLASS_COVERAGE=pd.DataFrame(CLASS_COVERAGE); SET_RUNS=pd.DataFrame(SET_RUNS)
CLASS_COVERAGE.to_csv(RUN_ROOT/'class_coverage.csv',index=False); SET_RUNS.to_csv(RUN_ROOT/'set_metrics_by_run.csv',index=False)
SET_SUMMARY=SET_RUNS.groupby(['cohort','spec','model','method'])[['coverage','mean_set_size','full_set_rate','singleton_rate']].mean().reset_index()
SET_SUMMARY.to_csv(RUN_ROOT/'set_summary.csv',index=False)
display(SET_SUMMARY.query('spec==@PRIMARY_SPEC and model in ["Coherent","Amplitude","SelectedClassical"]').round(3))

### Cell 28 — Compression, norm clipping and optimization diagnostics

Reports training-only PCA variance and held-out norm clipping. Histories flag selected
epochs at the budget boundary and optimizer convergence warnings. These flags identify
unresolved optimization limitations; they do not trigger automatic test-informed tuning.

In [ ]:
DIAGNOSTIC_ROWS=[]; OPTIMIZATION_ROWS=[]
def load_bundle(path): return joblib.load(path)['payload']
for item in BUNDLE_INDEX:
    bundle=load_bundle(item['path']); ds=DATASETS[item['cohort']]
    rep=bundle['rep']; idx=bundle['split']['test']; data=represent(ds['features'][rep['view']],rep)
    DIAGNOSTIC_ROWS.append(dict(**{k:item[k] for k in ['cohort','spec','repeat','fold','seed']},
        pca_rank=rep['rank'],requested_dimension=rep['dimension'],explained_variance=rep['explained_variance'],
        clipped_test_norm_fraction=float(((data['a'][idx]<=0)|(data['a'][idx]>=np.pi)).mean())))
    for name,rec in bundle['models'].items():
        OPTIMIZATION_ROWS.append(dict(**{k:item[k] for k in ['cohort','spec','repeat','fold','seed']},model=name,
            selected_epoch=rec.get('selected_epoch'),budget_exhausted=rec.get('budget_exhausted'),
            optimizer_converged=rec.get('converged'),selection_macro_f1=rec['score'][0],
            selected_parameters=json.dumps(rec.get('parameters',{}))))
pd.DataFrame(DIAGNOSTIC_ROWS).to_csv(RUN_ROOT/'representation_diagnostics.csv',index=False)
pd.DataFrame(OPTIMIZATION_ROWS).to_csv(RUN_ROOT/'optimization_diagnostics.csv',index=False)
display(SUMMARY.query('model in ["Coherent","Amplitude","PCA_SVM","QuadraticLift_LR","Uncompressed_LR"]')[['cohort','spec','model','macro_f1']].round(4))

### Cell 29 — Border-only shortcut screen on each outer fold

This uses image borders, dimensions and aspect ratio only. High predictive performance
suggests acquisition/formatting shortcuts worth investigating; it does not identify a causal
mechanism. Compare the foreground sensitivity representation with the unmasked version.

In [ ]:
SHORTCUT_ROWS=[]
for cohort,ds in DATASETS.items():
    frame=ds['manifest']; y=frame.label_id.to_numpy()
    for plan in SPLITS[cohort]:
        fi=capped_fit(frame,plan['split']['fit'],SPLIT_SEED+plan['repeat']*100+plan['fold']); ti=plan['split']['test']
        clf=RandomForestClassifier(n_estimators=100,max_depth=4,random_state=13,n_jobs=2)
        clf.fit(ds['border'][fi],y[fi],sample_weight=training_weights(frame,fi,ds['k']))
        tab=aggregate_units(frame,ti,clf.predict_proba(ds['border'][ti]))
        SHORTCUT_ROWS.append(dict(cohort=cohort,repeat=plan['repeat'],fold=plan['fold'],**metrics(tab.label_id,unit_probs(tab,ds['k']))))
SHORTCUTS=pd.DataFrame(SHORTCUT_ROWS); SHORTCUTS.to_csv(RUN_ROOT/'border_shortcut_metrics.csv',index=False)
display(SHORTCUTS.groupby('cohort').macro_f1.agg(['mean','std','min','max']).round(3))

### Cell 30 — Fixed acquisition perturbations

Uses the first outer fold and first initialization only, with the primary representation.
No corruption parameter is tuned, and no temperature or conformal threshold is refitted.
Patient-level results here are stress diagnostics, not independent scanner validation.

In [ ]:
STRESS_ROWS=[]
if RUN_DIAGNOSTICS:
    for cohort,ds in DATASETS.items():
        item=next(x for x in BUNDLE_INDEX if x['cohort']==cohort and x['spec']==PRIMARY_SPEC)
        bundle=load_bundle(item['path']); idx=bundle['split']['test']; frame=ds['manifest'].loc[idx].reset_index(drop=True)
        small=dict(ds,manifest=frame)
        chosen=['Amplitude','Passive','Coherent',bundle['selected_classical']]
        # CNN image perturbation requires its own image loader; use the fixed shared-feature controls here.
        chosen=[n for n in dict.fromkeys(chosen) if n!='FineTunedResNet18']
        for corruption in ['clean','noise','blur','bias','contrast']:
            X=features_for(frame,bundle['rep']['view'],corruption); data=represent(X,bundle['rep'])
            for name in chosen:
                rec=bundle['models'][name]; logits=slice_logits(rec,data,frame,np.arange(len(frame)),'test')
                tab=aggregate_units(frame,np.arange(len(frame)),softmax(logits/rec['temperature'],axis=1)); p=unit_probs(tab,ds['k'])
                STRESS_ROWS.append(dict(cohort=cohort,corruption=corruption,model=name,**metrics(tab.label_id,p),
                    **set_metrics(tab.label_id,conformal_sets(p,rec['conformal']['LAC']))))
STRESS=pd.DataFrame(STRESS_ROWS); STRESS.to_csv(RUN_ROOT/'perturbation_metrics.csv',index=False)
if len(STRESS): display(STRESS[['cohort','corruption','model','macro_f1','coverage','mean_set_size']].round(3))

### Cell 31 — Quantum gate resources and noise/shot diagnostics

Includes Möttönen amplitude preparation, not just the variational layer. Noise acts after
every decomposed gate. Joint sampling preserves covariance of commuting Z readouts. This is
a generic depolarizing simulation with one selected slice per patient, no noisy retraining,
and a small descriptive subset; it is neither hardware evidence nor a scaling advantage.

In [ ]:
def elementary_ops(model,d,a,weights):
    q=weights.shape[1]
    with qml.QueuingManager.stop_recording():
        d=np.asarray(d,dtype=np.complex128)
        if d.ndim!=1 or len(d)!=2**q or not np.isfinite(d).all(): raise ValueError('Invalid amplitude state for decomposition.')
        norm=float(np.linalg.norm(d))
        if norm<1e-12: raise ValueError('Cannot prepare a zero state.')
        d=d/norm
        prep=qml.MottonenStatePreparation.compute_decomposition(d,wires=list(range(q))); ops=list(prep)
        if model!='Amplitude': ops.append(qml.RY(np.pi/2 if model=='ConstantNorm' else a,wires=q))
        for axis,wire,value,_ in circuit_ops(model,weights):
            if axis=='CNOT': ops.append(qml.CNOT(wires=[wire,value]))
            elif axis=='H': ops.append(qml.Hadamard(wires=wire))
            elif axis=='Y': ops.append(qml.RY(value,wires=wire))
            else: ops.append(qml.RZ(value,wires=wire))
    return ops,len(prep)
def noisy_joint(model,d,a,weights,noise):
    n=weights.shape[1]+int(model!='Amplitude'); ops,_=elementary_ops(model,d,a,weights)
    @qml.qnode(qml.device('default.mixed',wires=n))
    def circuit():
        for op in ops:
            qml.apply(op)
            if noise:
                for wire in op.wires: qml.DepolarizingChannel(noise,wires=wire)
        return qml.probs(wires=range(n))
    p=np.maximum(np.asarray(circuit(),float),0); return p/p.sum()
RESOURCE_ROWS=[]; NOISE_ROWS=[]
for cohort,ds in DATASETS.items():
    item=next(x for x in BUNDLE_INDEX if x['cohort']==cohort and x['spec']==PRIMARY_SPEC)
    bundle=load_bundle(item['path']); data=represent(ds['features'][bundle['rep']['view']],bundle['rep'])
    frame=ds['manifest']; test=frame.loc[bundle['split']['test']].groupby('analysis_unit').head(1)
    per_class=max(1,CFG['noise_units']//ds['k']); sample=test.groupby('label_id',group_keys=False).head(per_class)
    for name in ['Amplitude','Passive','Coherent']:
        rec=bundle['models'][name]; q=rec['weights'].shape[1]; n=q+int(name!='Amplitude'); index=int(sample.index[0])
        ops,prep=elementary_ops(name,data['d'][index],data['a'][index],rec['weights'])
        levels=[0]*n
        for op in ops:
            if not op.wires: continue
            depth=1+max(levels[int(w)] for w in op.wires)
            for wire in op.wires: levels[int(wire)]=depth
        RESOURCE_ROWS.append(dict(cohort=cohort,model=name,wires=n,trainable_angles=rec['weights'].size,
            gates=len(ops),cnots=sum(op.name=='CNOT' for op in ops),depth=max(levels),preparation_operations=prep,
            note='algorithm-level un-routed count; no hardware calibration'))
        if not RUN_NOISE: continue
        for noise in ([0,0.01] if PROFILE=='smoke' else [0,0.001,0.005]):
            joint=np.stack([noisy_joint(name,data['d'][i],data['a'][i],rec['weights'],noise) for i in sample.index])
            signs=readout_signs(n,ds['k'])
            if noise==0:
                direct=quantum_logits(data['d'][sample.index],data['a'][sample.index],rec['weights'],name,ds['k'])
                error=float(np.max(abs(joint@signs.T-direct)))
                assert error<1e-8,f'Noise-free gate decomposition mismatch: {name} error={error}'
            for shots in ([None,128] if PROFILE=='smoke' else [None,128,1024,4096]):
                for repetition in range(1 if shots is None else CFG['shot_reps']):
                    rng=np.random.default_rng(81+repetition)
                    frequency=joint if shots is None else np.stack([rng.multinomial(shots,p)/shots for p in joint])
                    p=softmax((frequency@signs.T)/rec['temperature'],axis=1)
                    NOISE_ROWS.append(dict(cohort=cohort,model=name,noise=noise,shots='analytic' if shots is None else shots,
                        repetition=repetition,scope='one-slice-per-patient subset; diagnostic',**metrics(sample.label_id,p)))
pd.DataFrame(RESOURCE_ROWS).to_csv(RUN_ROOT/'circuit_resources.csv',index=False)
pd.DataFrame(NOISE_ROWS).to_csv(RUN_ROOT/'noise_and_shots.csv',index=False)
display(pd.DataFrame(RESOURCE_ROWS))

## Fresh external validation — optional data, required for a new external claim

### Cell 32 — Freeze final development models before external inspection

Uses the first main patient-complete cohort and the fixed primary representation. A final
fit/selection/temperature/conformal partition is made within development data. The first
initialization seed defines the external comparison prospectively; it is not picked from
internal test performance. The model identity is written before external results are read.

In [ ]:
FINAL_BUNDLE=None; EXTERNAL_STATUS={'status':'not_supplied','reason':'No independent external cohort supplied.'}
if EXTERNAL_ROOT:
    main=next(ds for name,ds in DATASETS.items() if name!='kaggle4')
    split=partition_development(main['manifest'],units_of(main['manifest']).analysis_unit,SPLIT_SEED+90001)
    split['test']=np.array([],int)
    final_plan=dict(repeat=0,fold=0,split=split)
    FINAL_BUNDLE=fit_bundle(main,PRIMARY_SPEC,final_plan,CFG['seeds'][0],stage='external_final')
    write_json(RUN_ROOT/'external_model_lock.json',dict(identity=FINAL_BUNDLE['identity'],protocol=PROTOCOL_HASH,
        selected_classical=FINAL_BUNDLE['selected_classical'],seed=CFG['seeds'][0],spec=PRIMARY_SPEC,
        lock_time_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())))
    print('External model locked:',FINAL_BUNDLE['identity'])
else: print(EXTERNAL_STATUS['reason'])

### Cell 33 — External provenance and overlap audit

Known patient matches, exact pixels and high-similarity images block evaluation. Automatic
checks cannot prove independence: the source custodian must verify patient linkage, independent
acquisition and that the cohort was not previously used for model decisions. Matching diagnostic
labels/reference standards are required. The Kaggle source must not be reused as external data.
A persistent registry blocks reuse of previously opened external patients with changed models
within the same output root. It cannot detect earlier uses elsewhere; changing folders does
not restore scientific independence.

In [ ]:
EXTERNAL_DS=None
if EXTERNAL_ROOT:
    if not EXTERNAL_METADATA_CSV or not EXTERNAL_PROVENANCE_JSON:
        raise ValueError('External evaluation requires metadata CSV and provenance JSON from Cell 9.')
    provenance=json.loads(Path(EXTERNAL_PROVENANCE_JSON).read_text(encoding='utf-8'))
    if provenance.get('synthetic_fixture') and PROFILE!='smoke':
        raise ValueError('Synthetic external fixtures are engineering checks only and cannot be evaluated as real paper evidence.')
    required=['institution','dataset_version','license','patient_id_namespace','independence_justification']
    if any(not provenance.get(k) or 'FILL_' in str(provenance[k]) for k in required): raise ValueError('Complete external provenance from verified source records.')
    if not all(provenance.get(k) is True for k in ['independent_acquisition','patient_linkage_checked','public_source_overlap_checked']):
        raise ValueError('External independence, patient linkage and public-source overlap checks require verified attestations.')
    if provenance.get('previously_used_for_model_decisions') is not False: raise ValueError('This external cohort has already influenced model decisions or its status is unknown.')
    folder=RUN_ROOT/'external'; raw=scan_folder(EXTERNAL_ROOT,EXTERNAL_METADATA_CSV)
    frame,labels,audit=audit_manifest(raw,folder)
    if not audit['complete_patient_unit_claim']: raise ValueError('External patient IDs must be complete and independently identifiable.')
    if labels!=main['labels']: raise ValueError(f'External classes {labels} differ from the target classes {main["labels"]}.')
    if 'site_id' not in frame: frame['site_id']='unknown'
    known=pd.concat([ds['manifest'] for ds in DATASETS.values()],ignore_index=True)
    pids=set(known.patient_id)-{''}; pixels=set(known.pixel_sha); tree=BKTree(); thumbs=[]
    for i,row in known.iterrows():
        tree.add(int(row.phash,16),i)
        thumbs.append((resize_float(normalize_image(read_image_row(row)),64)*255).astype('uint8'))
    overlaps=[]
    for _,row in frame.iterrows():
        reason=[]
        if row.patient_id in pids: reason.append('known_patient')
        if row.pixel_sha in pixels: reason.append('exact_pixel')
        gray=(resize_float(normalize_image(read_image_row(row)),64)*255).astype('uint8')
        for j in tree.search(int(row.phash,16),PHASH_RADIUS):
            if structural_similarity(gray,thumbs[j],data_range=255)>=NEAR_SSIM_THRESHOLD:
                reason.append('near_duplicate'); break
        if reason: overlaps.append(dict(relative_path=row.relative_path,analysis_unit=row.analysis_unit,reason=';'.join(reason)))
    pd.DataFrame(overlaps,columns=['relative_path','analysis_unit','reason']).to_csv(folder/'overlap_review.csv',index=False)
    write_json(folder/'provenance.json',provenance)
    if overlaps:
        EXTERNAL_STATUS=dict(status='blocked',reason=f'{len(overlaps)} overlapping or suspicious images. Resolve provenance; no external scores generated.')
    else:
        view=FINAL_BUNDLE['rep']['view']
        EXTERNAL_DS=dict(name='external',manifest=frame.reset_index(drop=True),labels=labels,k=len(labels),audit=audit,folder=folder,
            fingerprint=digest(frame.file_sha.tolist()),features={view:features_for(frame,view)})
        EXTERNAL_STATUS=dict(status='audit_passed',reason='Automated overlap checks passed; independence still relies on verified source records.')
        registry_path=Path(RESULTS_ROOT)/'external_evaluation_registry.json'
        registry=json.loads(registry_path.read_text(encoding='utf-8')) if registry_path.exists() else []
        patient_hashes={digest(['patient',p]) for p in frame.patient_id.unique()}
        pixel_hashes=set(frame.pixel_sha)
        previously_opened=[r for r in registry if patient_hashes.intersection(r['patient_hashes']) or pixel_hashes.intersection(r['pixel_hashes'])]
        if any(r['model_identity']!=FINAL_BUNDLE['identity'] for r in previously_opened):
            EXTERNAL_STATUS=dict(status='blocked_test_reuse',reason='External patients/images were opened with a different model in this output root. A new model requires fresh confirmation patients; changing the protocol hash does not reset exposure.')
            EXTERNAL_DS=None
        elif not previously_opened:
            registry.append(dict(model_identity=FINAL_BUNDLE['identity'],protocol=PROTOCOL_HASH,
                patient_hashes=sorted(patient_hashes),pixel_hashes=sorted(pixel_hashes),
                status='opened_for_evaluation',timestamp=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())))
            write_json(registry_path,registry)
    print(EXTERNAL_STATUS)

### Cell 34 — External metrics and fixed paired comparisons

No external labels influence fitting, model selection, preprocessing or calibration.
Two-sided paired patient permutation tests are Holm-corrected across the three fixed
comparisons. External coverage is empirical under possible shift; no in-distribution coverage
guarantee is asserted. A positive result is not required for a valid benchmark paper.

In [ ]:
EXTERNAL_METRICS=pd.DataFrame(); EXTERNAL_EFFECTS=pd.DataFrame()
def holm(p):
    p=np.array(p,float); order=np.argsort(p); out=np.empty(len(p))
    out[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order])); return out
if EXTERNAL_DS is not None:
    result=evaluate_bundle(EXTERNAL_DS,FINAL_BUNDLE,np.arange(len(EXTERNAL_DS['manifest'])),'external')
    for name,table in result.items(): table.to_csv(EXTERNAL_DS['folder']/f'{name}.csv',index=False)
    EXTERNAL_METRICS=result['metrics']; pred=result['predictions']; rows=[]; k=EXTERNAL_DS['k']
    a=pred[pred.model=='Coherent'].sort_values('analysis_unit'); y=a.label_id.to_numpy(); pa=unit_probs(a,k)[None]
    for comparator in ['Passive','Amplitude','SelectedClassical']:
        b=pred[pred.model==comparator].sort_values('analysis_unit')
        assert list(a.analysis_unit)==list(b.analysis_unit)
        pb=unit_probs(b,k)[None]; effect=paired_interval(y,pa,pb,CFG['bootstrap']); rng=np.random.default_rng(917); null=[]
        for _ in range(CFG['permutations']):
            swap=rng.random(len(y))<.5
            qa=np.where(swap[None,:,None],pb,pa); qb=np.where(swap[None,:,None],pa,pb)
            null.append(mean_f1(y,qa)-mean_f1(y,qb))
        p=(1+np.sum(np.abs(null)>=abs(effect['delta'])-1e-12))/(1+len(null))
        rows.append(dict(comparator=comparator,**effect,p_value=p,patients=len(y)))
    EXTERNAL_EFFECTS=pd.DataFrame(rows); EXTERNAL_EFFECTS['holm_p']=holm(EXTERNAL_EFFECTS.p_value)
    EXTERNAL_EFFECTS.to_csv(EXTERNAL_DS['folder']/'paired_effects.csv',index=False)
    EXTERNAL_STATUS=dict(status='executed',patients=len(y),scientific_evidence=PROFILE=='paper',
        reason='Fixed-model external evaluation; eligibility depends on truthful provenance and reference-standard review.')
    display(EXTERNAL_METRICS[['model','macro_f1','accuracy','nll']].round(4)); display(EXTERNAL_EFFECTS.round(4))
write_json(RUN_ROOT/'external_status.json',EXTERNAL_STATUS)

### Cell 35 — Figures with honest uncertainty labels

Primary bars show mean OOF macro-F1 and variation across repeat/seed runs. Effect intervals
are labelled exploratory. Coverage plots use one repeat and one seed so each patient appears
once. All figures and source tables are saved; no old result is copied into this notebook.

In [ ]:
for cohort in DATASETS:
    table=SUMMARY.query('cohort==@cohort and spec==@PRIMARY_SPEC').set_index('model')
    names=[n for n in ['Amplitude','Passive','Coherent','ConstantNorm','ShuffledNorm','QuadraticLift_LR','Uncompressed_LR','FineTunedResNet18','SelectedClassical'] if n in table.index]
    fig,ax=plt.subplots(figsize=(9,5))
    ax.barh(names,table.loc[names,'macro_f1'],xerr=table.loc[names,'repeat_seed_sd'].fillna(0),
        color=['#bb6548' if n=='Coherent' else '#416b91' for n in names],capsize=3)
    ax.invert_yaxis(); ax.set_xlim(0,1); ax.set_xlabel('Macro-F1: mean ± repeat/seed SD (descriptive)')
    ax.set_title(f'{cohort} · {PRIMARY_SPEC} · exploratory patient/unit OOF')
    fig.tight_layout(); fig.savefig(RUN_ROOT/f'{cohort}_primary.png',dpi=180); fig.savefig(RUN_ROOT/f'{cohort}_primary.pdf'); plt.show(); plt.close(fig)
    effects=EFFECTS.query('cohort==@cohort and spec==@PRIMARY_SPEC')
    fig,ax=plt.subplots(figsize=(8,4))
    ax.hlines(effects.comparator,effects.ci_low,effects.ci_high,color='#bb6548',lw=2)
    ax.scatter(effects.delta,effects.comparator,color='#bb6548'); ax.axvline(0,color='black',lw=1)
    ax.set_xlabel('Coherent minus comparator macro-F1'); ax.set_title('Exploratory conditional OOF bootstrap · not confirmatory')
    fig.tight_layout(); fig.savefig(RUN_ROOT/f'{cohort}_effects.png',dpi=180); plt.show(); plt.close(fig)
    coverage_seed=CFG['seeds'][0]
    cov=CLASS_COVERAGE.query('cohort==@cohort and spec==@PRIMARY_SPEC and repeat==0 and seed==@coverage_seed and model=="Coherent" and method=="LAC"')
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].errorbar(cov.label,cov.coverage,yerr=[cov.coverage-cov.ci_low,cov.ci_high-cov.coverage],fmt='o',capsize=4)
    axes[0].axhline(1-ALPHA,color='black',ls='--'); axes[0].set_ylim(0,1.02); axes[0].set_ylabel('Class coverage · Wilson interval')
    eff=SET_SUMMARY.query('cohort==@cohort and spec==@PRIMARY_SPEC and method=="LAC" and model in ["Amplitude","Coherent","SelectedClassical"]')
    axes[1].bar(eff.model,eff.mean_set_size,color='#416b91'); axes[1].set_ylim(0,DATASETS[cohort]['k']); axes[1].set_ylabel('Mean prediction-set size')
    fig.suptitle(f'{cohort} · coverage uses repeat 0, seed {coverage_seed}; efficiency averages runs')
    for axis in axes: axis.tick_params(axis='x',rotation=15)
    fig.tight_layout(); fig.savefig(RUN_ROOT/f'{cohort}_reliability.png',dpi=180); plt.show(); plt.close(fig)

### Cell 36 — Reconciliation and remaining evidence requirements

These are evidence checks, not an acceptance certificate. Lack of quantum superiority is
not a validity failure. Missing fresh external evidence limits generalization/confirmation
claims; it does not automatically prohibit an honestly scoped internal benchmarking paper.
Human novelty, provenance and manuscript review cannot be completed by a boolean script.

In [ ]:
checks=[]
def check(name,passed,scope,reason): checks.append(dict(requirement=name,passed=bool(passed),scope=scope,reason=reason))
reconciliation=[]
for keys,tab in PREDICTIONS.groupby(['cohort','spec','repeat','fold','seed','model']):
    co,sp,re,fo,se,mo=keys
    saved=FOLD_METRICS.query('cohort==@co and spec==@sp and repeat==@re and fold==@fo and seed==@se and model==@mo').macro_f1.iloc[0]
    got=metrics(tab.label_id,unit_probs(tab,DATASETS[co]['k']))['macro_f1']
    reconciliation.append(abs(saved-got))
assert max(reconciliation)<1e-12
check('Real full-budget study',PROFILE=='paper','scientific evidence','Synthetic/pilot executions are engineering or preliminary evidence.')
check('Circuit and gradient validation',len(MATH_CHECKS)==10,'implementation','Independent PennyLane and numerical-gradient checks passed.')
check('Every planned fold completed',len(BUNDLE_INDEX)==len(TASKS),'implementation','Checkpointed fold/representation/seed tasks reconciled.')
check('Saved metrics match predictions',max(reconciliation)<1e-12,'implementation','Every fold/model F1 recomputed.')
check('Primary cohort true patient identities',next(iter(DATASETS.values()))['audit']['complete_patient_unit_claim'],'internal validity','Kaggle remains exploratory when IDs are unavailable.')
check('No finite-sample conformal fallback',RELIABILITY.finite_thresholds.all(),'reliability','Each calibration class has enough independent units for a finite empirical threshold.')
primary_sets=SET_SUMMARY.query('spec==@PRIMARY_SPEC and model=="Coherent" and method=="LAC"')
check('Coherent sets are not always full',bool((primary_sets.full_set_rate<1-1e-12).all()),'usefulness','Possible finite thresholds can still yield uninformative sets; report the observed efficiency.')
check('Full feature sensitivity study',set(SPEC_LIBRARY).issubset(SPECS),'mechanism','Includes resolution, dimension, foreground, raw intensity and shared learned features.')
check('Fine-tuned practical comparator',RUN_FINE_TUNE and 'FineTunedResNet18' in FOLD_METRICS.model.unique(),'practical comparison','Paper protocol includes native-image CNN fitting on separate patients.')
check('Independent external cohort evaluated',EXTERNAL_STATUS['status']=='executed' and PROFILE=='paper','external claims',EXTERNAL_STATUS.get('reason',''))
for name,value in MANUAL_REVIEW.items(): check(name,value,'human review','Complete and document this item with the research team; do not mark it merely because code ran.')
REQUIREMENTS=pd.DataFrame(checks); REQUIREMENTS.to_csv(RUN_ROOT/'requirements.csv',index=False)
display(REQUIREMENTS)
print('This table describes evidence completeness. It does not predict conference acceptance.')

### Cell 37 — Calibration sample-size planning

The rank threshold only determines whether nontrivial prediction sets are possible.
It is not a guarantee of useful sets. The normal-approximation counts below are illustrative
precision planning for coverage near 90%, per class, and should be refined for the target
cohort, expected attrition and clinical reference standard. They are not a mandatory venue rule.

In [ ]:
planning=[]
for n in [8,9,12,30,60,140]:
    rank=math.ceil((n+1)*(1-ALPHA)); lo,hi=wilson(round(.9*n),n)
    planning.append(dict(calibration_patients_per_class=n,quantile_rank=rank,forced_full_threshold=rank>n,
        illustrative_coverage_ci_low=lo,illustrative_coverage_ci_high=hi))
pd.DataFrame(planning).to_csv(RUN_ROOT/'calibration_planning.csv',index=False)
display(pd.DataFrame(planning).round(3))
for halfwidth in [.10,.05]:
    n=math.ceil(1.96**2*.9*.1/halfwidth**2)
    print(f'Approximate independent evaluation patients per class for 90% coverage ±{halfwidth:.0%}: {n}')

### Cell 38 — Evidence-backed summary and manuscript working draft

The text uses this run's computed tables. Smoke/pilot outputs are prominently labelled.
Do not submit the automatic draft unchanged. Authors must supply related work, source/reference
standard details, clinical context, permissions, authorship, discussion and venue formatting.

In [ ]:
status='SYNTHETIC ENGINEERING CHECK — NOT SCIENTIFIC RESULTS' if PROFILE=='smoke' else ('PRELIMINARY PILOT — NOT THE FULL STUDY' if PROFILE=='pilot' else 'REAL-DATA PAPER-PROFILE EXPERIMENT — AUTHOR REVIEW REQUIRED')
lines=['# Norm Information and Coherence in Quantum MRI Classification','',status,'',
 '## Research question',
 'Does coherent norm fusion provide a reproducible predictive or reliability benefit beyond amplitude encoding, passive fusion, norm ablations and classical controls?',
 '', '## Design',
 f'Primary representation: {PRIMARY_SPEC}. Outer folds: {CFG["folds"]}; repeats: {CFG["repeats"]}; primary seeds: {CFG["seeds"]}.',
 'All internal analyses are exploratory because the original public-cohort test results were inspected before this redesign.',
 'PCA and supervised training use fit patients only. Inner selection, temperature calibration, conformal calibration and outer evaluation use separate patients.',
 '', '## Observations']
for cohort,ds in DATASETS.items():
    t=SUMMARY.query('cohort==@cohort and spec==@PRIMARY_SPEC')
    lines.extend([f'### {cohort}',f'Retained slices: {len(ds["manifest"])}; unique analysis units: {ds["manifest"].analysis_unit.nunique()}; verified patient-ID coverage: {ds["audit"]["patient_coverage"]:.1%}; prediction unit: {ds["audit"]["prediction_unit"]}.',
        markdown_table(t[['model','macro_f1','repeat_seed_sd','accuracy','nll','brier']].round(4)),
        'Exploratory paired effects (conditional OOF bootstrap; dependent training sets):',
        markdown_table(EFFECTS.query('cohort==@cohort and spec==@PRIMARY_SPEC')[['comparator','delta','ci_low','ci_high']].round(4))])
    effect=EFFECTS.query('cohort==@cohort and spec==@PRIMARY_SPEC and comparator=="Passive"').iloc[0]
    lines.append(f'Coherent minus Passive macro-F1: {effect.delta:+.4f}. This is an exploratory estimate and does not establish a prospective superiority claim.')
    efficiency=SET_SUMMARY.query('cohort==@cohort and spec==@PRIMARY_SPEC and model=="Coherent" and method=="LAC"').iloc[0]
    lines.append(f'Coherent LAC mean coverage: {efficiency.coverage:.3f}; mean set size: {efficiency.mean_set_size:.3f}; full-set rate: {efficiency.full_set_rate:.3f}.')
lines.extend(['','## External evidence',json.dumps(EXTERNAL_STATUS,indent=2),'',
 '## Interpretation boundaries',
 '- No computational quantum advantage is established: these small circuits have exact quadratic classical inference equivalents.',
 '- A poor quantum result is retained. Do not interpret a non-significant comparison as proof of equivalence.',
 '- Internal bootstrap intervals omit refitting/institution uncertainty and are not confirmation tests.',
 '- Calibration assumes within-class exchangeability; external or simulated shifts can invalidate coverage guarantees.',
 '- Unknown Kaggle patient lineage prevents patient-generalization claims; public cohorts may share source images.',
 '- Foreground extraction and post-normalization perturbations are sensitivity tests, not validated clinical preprocessing or independent scanners.',
 '', '## Unresolved requirements',markdown_table(REQUIREMENTS.loc[~REQUIREMENTS.passed])])
text='\n\n'.join(lines)
(RUN_ROOT/'study_summary.md').write_text(text,encoding='utf-8')
draft='# Manuscript working draft\n\n'+status+'\n\n'
draft+='## Suggested title\n\nWhen Does Norm Information Help Small Quantum MRI Classifiers? A Patient-Level Benchmark\n\n'
draft+='## Abstract material\n\n'+text+'\n\n'
draft+='## Author completion tasks\n\nState the precise novel finding against the related-work table. Explain clinical reference standards and inclusion/exclusion flow. Include optimization, compression, shortcut and negative findings. Document source permissions/ethics accurately. Discuss limitations and complete author contributions, funding, conflicts and references.\n'
(RUN_ROOT/'manuscript_working_draft.md').write_text(draft,encoding='utf-8')
display(Markdown(text[:5000]))

### Cell 39 — References, model cards and reproducibility records

These sources support study design and existing methods; their existence does not verify
the novelty of this exact experiment. Recheck related work before submission. Do not compare
published headline accuracies across incompatible splits, taxonomies or evaluation units.

In [ ]:
REFERENCES=[
 {'title':'Better than classical? The subtle art of benchmarking quantum machine learning models','url':'https://arxiv.org/abs/2403.07059','role':'Fair comparisons and controlled negative findings'},
 {'title':'A quantum computing approach for brain tumor image classification using amplitude encoding technique','url':'https://doi.org/10.1007/s44464-026-00005-z','role':'Existing PCA/amplitude MRI approach'},
 {'title':'HQCM-EBTC: A Hybrid Quantum-Classical Model for Explainable Brain Tumor Classification','url':'https://arxiv.org/abs/2506.21937','role':'Related hybrid MRI preprint; do not treat hybrid architecture alone as new'},
 {'title':'A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification','url':'https://arxiv.org/abs/2107.07511','role':'Finite-sample calibration, APS, class-conditional coverage and exchangeability'},
 {'title':'Brain tumor dataset, version 8','url':'https://doi.org/10.6084/m9.figshare.1512427.v8','role':'Native patient/mask-bearing three-class source'},
 {'title':'Brain Tumor MRI Dataset','url':'https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset','role':'Optional composite four-class benchmark'},
 {'title':'TRIPOD+AI statement','url':'https://doi.org/10.1136/bmj-2023-078378','role':'Clinical prediction reporting guidance'},
 {'title':'Torchvision ResNet18 documentation','url':'https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html','role':'Pretrained weight specification'},
 {'title':'PennyLane AmplitudeEmbedding','url':'https://docs.pennylane.ai/en/stable/code/api/pennylane.AmplitudeEmbedding.html','role':'State preparation semantics; notebook uses pinned 0.42.3'},
]
write_json(RUN_ROOT/'references.json',REFERENCES)
(RUN_ROOT/'references.md').write_text('# References\n\n'+'\n\n'.join(f'- {r["title"]}. {r["url"]}\n  Role: {r["role"]}' for r in REFERENCES),encoding='utf-8')
cards=[]
for name,ds in DATASETS.items():
    cards.append(dict(cohort=name,classes=ds['labels'],purpose='retrospective research benchmark',
        patient_ID_coverage=ds['audit']['patient_coverage'],primary_representation=PRIMARY_SPEC,
        training_slice_cap=CFG['slice_cap'],prediction_unit=ds['audit']['prediction_unit'],aggregation='mean calibrated slice probabilities per analysis unit',
        prohibited_unsupported_claims=['clinical deployment readiness','quantum computational advantage','guaranteed conference acceptance'],
        external_status=EXTERNAL_STATUS,protocol=PROTOCOL_HASH))
write_json(RUN_ROOT/'model_cards.json',cards)
write_json(RUN_ROOT/'run_completion.json',dict(profile=PROFILE,planned_tasks=len(TASKS),completed_tasks=len(BUNDLE_INDEX),
    actual_results=True,scientific_full_run=PROFILE=='paper',external_status=EXTERNAL_STATUS,
    all_metrics_reconciled=True,implementation_checks_passed=True,acceptance_certified=False))
print('References, model cards and run completion record saved.')

### Cell 40 — Export the evidence bundle

The archive contains summaries, per-unit predictions, audit/split records and figures.
Raw images, pretrained weights and model checkpoints are excluded to keep the ZIP small;
checkpoints remain in the output directory for resuming. Metadata may still contain patient
identifiers or paths: review permissions before sharing. The notebook itself is the source
artifact; download it from Colab after running to preserve cell outputs.

In [ ]:
readme=f"""# QML MRI evidence
Profile: {PROFILE}
Protocol: {PROTOCOL_HASH}
Status: {status}
Read study_summary.md, requirements.csv and external_status.json first.
This archive does not certify novelty, clinical usefulness or publication acceptance.
Code source: the executed QML_MRI_Publication_Study_v3_fixed.ipynb; download it from Colab.
Model and candidate checkpoints remain under the run directory and are excluded here.
Internal OOF analyses are exploratory after prior public-test inspection.
Review identifiers, source permissions and licensing before sharing.
"""
(RUN_ROOT/'README.md').write_text(readme,encoding='utf-8')
ZIP_PATH=RUN_ROOT.parent/(RUN_ROOT.name+'_evidence.zip')
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.json','.md','.txt','.png','.pdf'}:
            archive.write(path,path.relative_to(RUN_ROOT))
print('Evidence ZIP:',ZIP_PATH)
print('Checkpoint directory:',RUN_ROOT)
from IPython.display import FileLink
display(FileLink(str(ZIP_PATH)))
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print('Outside Colab: download the ZIP using the link above or open the printed path.')
    else:
        files.download(str(ZIP_PATH))

## Takeaways and submission workflow

1. Read `study_summary.md` and `requirements.csv` after the real `paper` run.
2. Inspect the effect estimates and convergence/shortcut controls before choosing the paper's claims.
3. A negative result is retained. Adding cells or obtaining a small p-value does not establish novelty.
4. Internal experiments remain exploratory. Repeated evaluation does not create new independent patients.
5. Complete the literature comparison, data permissions and manuscript; obtain fresh external evidence
   if claiming generalization or new confirmation. Keep all protocol versions and test-access records.
6. Share the code and permitted evidence with the paper. Review patient metadata and source licenses
   before sharing any archive. The automatic manuscript is a working draft, not a submitted paper.